last modified date : 2026.05
제작 : 모두의연구소

---

## 이 노트북에 내가 더한 것 — 2026.07.27, 김민욱

위는 원본 실습 노트북(모두의연구소 제작)이고, 나는 거기 비어 있던 TODO 를 채운 뒤 몇 가지를 더 얹었다.
**원본 설명 문장은 손대지 않았고**, 내가 쓴 부분은 코드 주석과 `실험 기록` · `회고` 마크다운으로 구분해 뒀다.

**1. 빈칸 구현 (원본 과제)**
RAG-Fusion 의 RRF 함수, Self-RAG 프롬프트 2개(검색 필요성 판단 · 답변 자가 비평),
그리고 추가 실습인 KLUE-MRC 이식 Step B~K 전체.

**2. 평가 지표의 한국어 버그를 찾아 고침**
RAGAS 의 `answer_relevancy` 가 멀쩡한 한국어 답변에 **음수(-0.0166)** 를 주는 걸 발견했다.
답변에서 질문을 역추론할 때 영어 질문을 만들어내 언어가 어긋나는 것이 원인이었다.
역질문 프롬프트에 지시문 한 줄을 더해 교정했다. 이 패치 없이는 비교표의 그 열이 통째로 노이즈였다.

**3. 통계 검정 추가 (Step K)**
질문 20개 평균만 비교하면 우연을 실력으로 착각한다. 문항별 점수를 짝지어 **paired t-test** 를 돌려
"이 개선이 통계적으로 유의미한가" 를 두 도메인 모두에서 확인했다.

**4. 같은 코드를 3번 돌려 재현성을 확인**
`temperature=0` 인데도 실행마다 점수가 달라졌다. 어떤 지표가 흔들리고 어떤 지표가 안 흔들리는지
직접 비교했다 (`실험 기록 2`).

**5. Langfuse 를 붙여 데이터 흐름을 눈으로 보게 함**
관측 서버를 **Docker 로 내 우분투에 직접 띄우고**(컨테이너 6개), LangChain 객체는 전역 등록으로,
내가 쓴 파이썬 함수 9곳은 `@observe` 로 잡았다. 함수마다 인자와 리턴값이 기록돼서,
나중에 복습할 때 "어떤 문서가 들어가 어떤 답이 나왔는지" 를 화면에서 따라갈 수 있다.

**6. 실패한 과정도 남김**
Vector DB 두 개가 사실은 하나였던 사고를 잡아낸 과정을 지우지 않고 `실험 기록 1` 로 남겼다.
에러가 안 나는데 조용히 틀리는 종류라, 다시 볼 때 이쪽이 더 쓸모 있다고 생각했다.

**7. GitHub 에서 깨져 보이는 출력을 고침**
RAGAS 진행률 표시줄(ipywidgets)이 "Could not render" 경고로 뜨는 것을 발견해 그 출력을 걷어냈다.
내 화면과 남의 화면(GitHub)이 다를 수 있다는 걸 배웠다.

**실행 환경**: 콜랩 원본을 로컬로 옮겨 우분투에서 실행했다.
conda env `rag2` (ragas 0.2.10 / langchain 0.3.27 / chromadb 1.5.9), 리랭커는 ROCm GPU 사용.


# Day 2 실습 — Advanced·Modular RAG + RAGAS 평가

# 들어가며

Day 1에서는 가장 기본형인 **Naive RAG** 파이프라인을 직접 구현해 보았습니다. 이번 실습에서는 한국어 QA 벤치마크 **KorQuAD v1** 데이터셋 위에서 **Advanced·Modular RAG** 의 핵심 기법(Multi-Query, RAG-Fusion, HyDE, Reranking, Self-RAG)을 단계적으로 적용하고, 그 결과를 **RAGAS** 로 정량 평가합니다.

이번 실습이 끝나면 다음을 직접 말할 수 있게 됩니다.
- Naive RAG 대비 **어떤 단계**를 보강하면 정답률이 올라가는가
- Multi-Query / RAG-Fusion / HyDE / Reranker / Self-RAG 는 각각 **어떤 코드 라인**으로 적용하는가
- RAGAS 의 4대 지표(Faithfulness · Answer Relevance · Context Precision · Context Recall)는 어떻게 계산되고 어떻게 읽는가
- 내 RAG 가 ‘얼마나 좋아졌는지’를 **숫자로** 보여주는 방법

## Step 0 : 설치와 준비  
Day 1과 동일하게 Colab에서 진행한다고 가정합니다.

In [1]:
# [로컬 수정] 오늘 TensorFlow 는 한 줄도 안 쓰는데, transformers 가 "TF가 설치돼 있네" 하고
# 자동으로 끌어온다. 그런데 이 환경은 protobuf 7.35 와 tensorflow 2.19 궁합이 안 맞아서
# 빨간 AttributeError 가 5줄씩 찍힌다. (실행은 성공하지만 노트북이 지저분해짐)
# 그래서 아예 부르지 않도록 막는다. ★ 반드시 다른 import 보다 먼저 실행돼야 한다.
import os
os.environ["USE_TF"] = "0"          # transformers 에게 "TF 쓰지 마"
os.environ["USE_JAX"] = "0"         # JAX 도 마찬가지
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # 혹시 로드돼도 로그는 조용히

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)   # langchain-community sunset 안내 끄기

print("TF 차단 설정 완료")

''' 이 셀을 설정하지 않으면 이런 에러 발생
---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
'''

TF 차단 설정 완료


" 이 셀을 설정하지 않으면 이런 에러 발생\n---------------------------------------------------------------------------\nAttributeError                            Traceback (most recent call last)\nAttributeError: 'MessageFactory' object has no attribute 'GetPrototype'\n"

In [2]:
# --- langfuse 연결 (내 우분투에 띄운 셀프호스트 서버) ---
# 이걸 켜두면 이 노트북에서 일어나는 일이 전부 웹 화면에 트리로 쌓인다.
# 나중에 복습할 때 "어떤 함수가 어떤 값을 받아 무엇을 돌려줬는지" 를 순서대로 볼 수 있다.
import os
from pathlib import Path
from contextvars import ContextVar
from typing import Optional

LANGFUSE_OK = False
try:
    cfg = Path.home() / ".config" / "langfuse"
    os.environ["LANGFUSE_PUBLIC_KEY"] = (cfg / "public_key").read_text().strip()
    os.environ["LANGFUSE_SECRET_KEY"] = (cfg / "secret_key").read_text().strip()
    os.environ["LANGFUSE_HOST"]       = "http://localhost:3000"   # 클라우드 아님

    from langfuse import get_client, observe
    from langfuse.langchain import CallbackHandler
    from langchain_core.tracers.context import register_configure_hook

    lf_client = get_client()
    handler   = CallbackHandler()

    # ★핵심 1: 핸들러를 LangChain 전역에 등록한다.
    #   이 뒤로는 config={"callbacks":[handler]} 를 안 써도 모든 체인·리트리버 호출이 자동 기록된다.
    #   (register_configure_hook 은 커널당 한 번만 — 두 번 부르면 트레이스가 중복될 수 있다)
    lf_var: ContextVar[Optional[CallbackHandler]] = ContextVar("lf_handler", default=None)
    if not globals().get("_LF_HOOK_REGISTERED"):
        register_configure_hook(lf_var, True)
        _LF_HOOK_REGISTERED = True
    lf_var.set(handler)

    LANGFUSE_OK = lf_client.auth_check()
    print("langfuse 연결:", LANGFUSE_OK)   # True 나와야 함
except Exception as e:
    print("[langfuse 연결 실패 — 트레이싱 없이 계속 진행한다]", e)

# ★핵심 2: 전역 등록으로 잡히는 건 'LangChain 객체' 뿐이다.
#   내가 직접 쓴 파이썬 함수(RRF 계산, rerank, advanced_rag ...)는 LangChain 이 모르니 안 잡힌다.
#   그래서 아래부터 그 함수들에 @observe 를 한 줄씩 붙인다.
#   단, langfuse 연결이 실패했을 때도 노트북이 돌아가야 하므로 아무 일도 안 하는 대체품을 만들어 둔다.
if not LANGFUSE_OK:
    import contextlib

    def observe(*dargs, **dkwargs):
        """langfuse 가 없을 때 쓰는 가짜 데코레이터. 함수를 그대로 돌려준다."""
        if dargs and callable(dargs[0]) and not dkwargs:
            return dargs[0]              # @observe 처럼 괄호 없이 쓴 경우
        def deco(fn):
            return fn                    # @observe(name="...") 처럼 쓴 경우
        return deco

    class _NoLangfuse:
        @contextlib.contextmanager
        def start_as_current_observation(self, *a, **k):
            class _Span:
                def update(self, *a, **k): pass
            yield _Span()
        def flush(self): pass

    lf_client = _NoLangfuse()
    print("(langfuse 없이 진행 — @observe 는 아무 일도 하지 않는 껍데기로 동작한다)")


langfuse 연결: True


### 이 노트북을 나중에 다시 볼 때 — Langfuse 로 데이터 흐름 따라가기

(오늘 미션의 [사이드 프로젝트] Langfuse 적용 부분이다.)

코드를 다시 읽는 것만으로는 **값이 어디서 어디로 흘러갔는지**가 잘 안 잡힌다.
그래서 이 노트북은 실행되는 동안 모든 단계를 Langfuse 서버로 보내도록 해뒀다.
나중에 복습할 때 웹 화면에서 트리로 펼쳐 보면 된다.

**서버는 클라우드가 아니라 내 우분투에 Docker 로 직접 띄웠다.**
Langfuse 클라우드 무료 플랜을 써도 되지만, 학습 데이터가 밖으로 나가지 않게 하고 싶었고
컨테이너 구성을 직접 보고 싶어서 셀프호스팅을 골랐다. `docker compose up -d` 로
**컨테이너 6개**(웹 서버, 워커, PostgreSQL, ClickHouse, Redis, MinIO)가 뜬다.
관측 데이터가 왜 이렇게 여러 저장소를 쓰는지도 이때 알게 됐다 — 트레이스 같은 대량 이벤트는
ClickHouse(분석용 컬럼 저장소)에, 계정·프로젝트 같은 건 PostgreSQL 에 나눠 넣는다.

띄우면서 걸린 것 하나만 적어두면, ClickHouse 가 쓰려는 호스트 포트 9000 이 이미 물려 있어서
컨테이너가 안 떴는데 **범인이 내 주피터 커널**이었다. 커널 하나가 포트를 5개(9000~9004)나
잡고 있었던 것이다. 호스트 쪽 포트만 19000 으로 바꿔서 해결했다
(컨테이너끼리는 내부망에서 여전히 9000 으로 통신하므로 앱은 멀쩡하다 — "호스트 포트와
컨테이너 포트는 다른 것" 이라는 걸 여기서 몸으로 알았다).

**두 가지를 서로 다른 방법으로 잡는다는 게 핵심이다.**

| 무엇 | 예 | 어떻게 잡히나 |
|---|---|---|
| LangChain 객체 | 체인, 리트리버, ChatOpenAI | 위 준비 셀의 **전역 등록** 한 번으로 자동 |
| 내가 쓴 파이썬 함수 | `reciprocal_rank_fusion`, `rerank`, `advanced_rag`, `self_rag` | **`@observe` 한 줄**을 직접 붙여야 함 |
| 함수로 안 쪼갠 코드 블록 | "후보 10개 -> 상위 3개" 구간 | **`with ...start_as_current_observation(as_type="retriever")`** |

처음에 전역 등록만 해두고 셀을 돌렸을 때 **LangChain 호출만 잡히고 내 함수는 하나도 안 보였다.**
LangChain 은 자기가 만든 객체만 아는데, RRF 계산 같은 건 그냥 파이썬 함수라 LangChain 이 모르기 때문이다.
그래서 함수마다 `@observe(name=...)` 를 붙였다. 이걸 붙이면 **인자와 리턴값이 자동으로 기록된다** —
어떤 문서 리스트가 들어가서 어떤 3개가 나왔는지가 화면에 그대로 남는다.

**이름 앞에 번호를 붙여뒀다.** 화면에서 실행 순서대로 읽히라고 그렇게 했다.

```
01_질문확장(Multi-Query)      02_RRF융합         03_HyDE검색
04_리랭킹(CrossEncoder)       05_Advanced_RAG(전체)
10_Self-RAG(판단+비평+재시도)
21_HyDE검색(KLUE)             22_리랭킹(KLUE)    23_Advanced_RAG_KLUE(전체)
```

**계층은 저절로 잡힌다.** 바깥 함수에 `@observe` 가 걸려 있으면 그 안에서 일어난 모든 것이
자식으로 붙는다 (인자로 뭘 넘기지 않아도 된다. SDK 가 "지금 열려 있는 단계" 를 따라다니기 때문).
그래서 `advanced_rag` 하나를 펼치면 이런 모양이 된다.

```
05_Advanced_RAG(전체)              (SPAN)   <- @observe
├─ Chroma 검색                     (RETRIEVER)  <- LangChain 자동
├─ 후보10개→상위3개                (RETRIEVER)  <- with 블록. 근거 문서 본문이 남는다
│  └─ 04_리랭킹(CrossEncoder)      (SPAN)       <- @observe
└─ RunnableSequence                (CHAIN)      <- LangChain 자동
   └─ ChatOpenAI                   (GENERATION) <- 토큰/비용/시간까지 자동
```

**복습할 때 이렇게 보면 좋다** (내가 다시 볼 때를 위해 적어둠)

1. `후보10개→상위3개` 를 펼쳐 **근거로 들어간 문서 3개의 본문**을 읽는다.
2. 그다음 `ChatOpenAI` 의 출력(최종 답변)과 대조한다.
   -> **"문서 3개를 줬는데 답에 실제로 쓰인 건 몇 개인가?"** 를 눈으로 확인할 수 있다.
   context_precision 이 무엇을 재는 지표인지 숫자가 아니라 화면으로 이해된다.
3. `10_Self-RAG` 를 펼치면 판정(YES/NO) -> 답변 -> 자가 비평(SUPPORTED/NOT_SUPPORTED) -> 재시도가
   순서대로 나온다. 이 노트북에서 비평이 맞는 답을 NOT_SUPPORTED 로 판정한 장면도 여기서 확인된다.
4. KorQuAD(`05_`)와 KLUE(`23_`)를 나란히 열면 **같은 구조에 도메인만 다른 것**이 보인다.

**서버가 꺼져 있어도 이 노트북은 그대로 돌아간다.** 준비 셀에서 연결에 실패하면
`@observe` 를 아무 일도 하지 않는 껍데기로 바꿔 두기 때문이다.
(관측은 어디까지나 곁다리인데, 그것 때문에 실습이 멈추면 곤란하다.)


In [3]:
# ── 로컬(rag2 env) 전용 ──────────────────────────────────────────────
# 콜랩 원본은 여기서 langchain 을 0.2.x 로 다운그레이드했지만,
# 우리는 rag2 env 에 호환 조합을 미리 깔아뒀다. 그래서 설치는 하지 않는다.
#   (설치를 다시 돌리면 오히려 transformers/리랭커가 깨진다 — LOG.md 07-24 참고)
# 대신 버전만 눈으로 확인하고 지나간다.
import ragas, langchain, langchain_core, chromadb, transformers
print("ragas       ", ragas.__version__)          # 0.2.10
print("langchain   ", langchain.__version__)        # 0.3.x
print("langchain_core", langchain_core.__version__)
print("chromadb    ", chromadb.__version__)          # 1.5.9
print("transformers", transformers.__version__)      # 4.57.6 (리랭커용)


ragas        0.2.10
langchain    0.3.27
langchain_core 0.3.79
chromadb     1.5.9
transformers 4.57.6


> (로컬 메모) 콜랩에서는 위 설치 셀 실행 뒤 런타임을 재시작해야 했지만,
> 로컬 rag2 커널에서는 그럴 필요가 없다. 위 셀은 버전 확인만 하니 그냥 지나가면 된다.


In [4]:
import os
# chromadb 익명 통계 전송 끄기 — posthog SDK 인자 충돌로 ERROR 로그가 뜨는 것 방지
os.environ["ANONYMIZED_TELEMETRY"] = "False"
# 어제 배운 것: protobuf vs tensorflow 노이즈 차단 (빨간 글씨 != 실패)
os.environ["USE_TF"] = "0"

import nest_asyncio
nest_asyncio.apply()  # RAGAS 가 주피터의 비동기 이벤트 루프와 충돌하지 않도록


In [5]:
# 콜랩은 userdata.get("OPENAI_KEY") 로 키를 읽었지만,
# 로컬에서는 어제 저장해둔 파일에서 읽는다. (권한 600, git 에 안 올라감)
import os
os.environ["OPENAI_API_KEY"] = open(
    os.path.expanduser("~/.config/openai/api_key")).read().strip()
print("OPENAI_API_KEY 등록됨:", os.environ["OPENAI_API_KEY"][:7] + "..." )


OPENAI_API_KEY 등록됨: sk-proj...


## Step 1 : KorQuAD v1 위에서 Naive RAG 베이스라인 만들기

Day 1에서 만든 RAG 파이프라인을 한국어 QA 벤치마크 **KorQuAD v1** 위에 다시 한 번 올립니다. 이후 단계는 모두 이 베이스라인 위에 ‘덧붙이는’ 방식입니다.

- HuggingFace `datasets` 로 KorQuAD v1 자동 다운로드 (별도 PDF 업로드 불필요)
- 일부만 샘플링해 토큰 비용 통제 (unique context 약 200개)
- Embedding → VectorStore → Retriever → LLM
- 검색 전략은 단순 `similarity` (top-k)

**📥 데이터셋**: <https://huggingface.co/datasets/KorQuAD/squad_kor_v1>

In [6]:
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import tiktoken, random

tokenizer = tiktoken.get_encoding("cl100k_base")
def tiktoken_len(text):
    return len(tokenizer.encode(text))

# 1) 데이터셋 로드 + 2000개 샘플링 + context 중복 제거 → unique 약 800개
#    (Vector DB 가 크면 Reranker 의 정밀도 개선 효과가 더 또렷하게 보입니다.
#     인덱싱 토큰 비용 약 0.01 USD 추가)
raw_ds = load_dataset("squad_kor_v1", split="validation").shuffle(seed=42).select(range(2000))

unique = {}
for ex in raw_ds:
    if ex["context"] not in unique:
        unique[ex["context"]] = ex["title"]
context_docs = [Document(page_content=c, metadata={"title": t}) for c, t in unique.items()]

# 2) chunk 단위로 분할 (KorQuAD context는 짧지만 길이 균질화를 위해 splitter 사용)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function=tiktoken_len)
docs = splitter.split_documents(context_docs)

# 3) Embedding & Chroma 적재 — chunk 약 800개를 한 번에 넣으면 chromadb 의 batch limit
#    (Colab 환경에서 보통 5461) 또는 OpenAI rate limit 에 걸릴 수 있어
#    100개씩 배치로 add_documents 합니다.
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma(embedding_function=embedding)
BATCH = 100
for i in range(0, len(docs), BATCH):
    db.add_documents(docs[i:i+BATCH])

# 4) Retriever (Naive: similarity)
naive_retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# 5) LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0)
print(f"베이스라인 준비 완료 — unique context: {len(context_docs)}, chunks: {len(docs)}")

베이스라인 준비 완료 — unique context: 847, chunks: 1264


베이스라인 RAG로 간단한 질의를 던져 답이 나오는지 확인합니다.

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT = ChatPromptTemplate.from_template(
    "다음 문서를 참고해 질문에 한국어로 간결하게 답하세요. 문서에 없는 내용은 만들지 마세요.\n\n"
    "[문서]\n{context}\n\n"
    "[질문]\n{question}\n\n"
    "[답변]"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

naive_chain = (
    {"context": naive_retriever | format_docs,
     "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# 데이터셋에서 첫 질문 하나를 뽑아 테스트
TEST_Q = raw_ds[0]["question"]
print("Q:", TEST_Q)
print("A:", naive_chain.invoke(TEST_Q))

Q: 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?


A: 대중교통체계입니다.


## Step 2 : Pre-retrieval 강화 — Multi-Query Retrieval  

사용자가 던진 질문 하나로만 검색하면 ‘다른 표현’으로 적힌 정답을 놓칠 수 있습니다. **Multi-Query Retrieval**은 LLM에게 ‘같은 의도의 다른 질문 N개’를 만들게 시킨 뒤, 각 질문으로 병렬 검색하고 결과를 합칩니다.

LangChain은 이를 한 클래스로 제공합니다.

In [8]:
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    llm=ChatOpenAI(model="gpt-4o", temperature=0),
)

# 어떤 ‘유사 질문’으로 확장되는지 로그로 확인 가능
docs_mq = multi_query_retriever.invoke(TEST_Q)
print(f"검색된 문서 수: {len(docs_mq)}")
print("---")
print(docs_mq[0].page_content[:300])

INFO:langchain.retrievers.multi_query:Generated queries: ['1. 2004년에 이명박 서울시장이 추진한 주요 도시 개선 프로젝트는 무엇인가요?', '2. 이명박이 서울시장으로 재직하던 2004년에 시행한 도시 발전 계획은 어떤 것들이 있나요?', '3. 2004년 이명박 서울시장이 주도한 서울시의 주요 변화나 개혁은 무엇이었나요?']


검색된 문서 수: 5
---
2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도 했다. 하지만 새 교통체계가 정착되면서 많은 긍정적인 효과를 가져오게 된다. 중앙버스차로 도입으로 버스의 평균 속도가 증가하여 정시에 도착하는 빈도가 늘어났고 환승제도로 인한 교


## Step 2.5 : RAG-Fusion — Multi-Query + RRF로 묶어내기

Day2_1 노트에서 “꼭 짚고 가라”고 했던 패턴 중 하나가 **RAG-Fusion** 입니다. Step 2의 Multi-Query는 ‘유사 질문 N개로 병렬 검색’ 까지만 했는데, **RAG-Fusion** 은 그 N개 검색 결과를 **Reciprocal Rank Fusion (RRF)** 라는 간단한 공식으로 합쳐 ‘여러 쿼리에서 공통으로 상위에 떴던 문서’ 를 최상단으로 끌어올립니다.

RRF 점수 공식:

$$
\text{score}(d) = \sum_{i=1}^{N} \frac{1}{k + \text{rank}_i(d)}
$$

- $\text{rank}_i(d)$ : i번째 쿼리의 결과에서 문서 $d$ 의 순위 (1부터)
- $k$ : 스무딩 상수 (관례적으로 60)

아래 셀에서는 (1) sub-query 생성, (2) 각 sub-query 로 검색, (3) **RRF 함수는 여러분이 직접 채우기**, (4) 결과 확인까지 한 번에 해봅니다.

In [9]:
from collections import defaultdict

# (1) sub-query 생성 — Multi-Query 가 내부적으로 하는 일을 명시적으로 노출 (한국어)
SUBQUERY_PROMPT = ChatPromptTemplate.from_template(
    "당신은 검색 보조 AI 입니다. 다음 질문과 의미는 같지만 표현이 다른 4개의 한국어 검색 쿼리를 만드세요. "
    "오직 4개의 쿼리만 한 줄에 하나씩 출력하고, 번호나 다른 설명은 붙이지 마세요.\n\n질문: {question}"
)

# ┌─ [관측용 · langfuse] ─ 여기부터 한 줄은 실습 로직이 아니다 ──────────────
# │  이 데코레이터는 함수가 하는 일을 바꾸지 않는다. 함수가 호출될 때
# │  '어떤 인자를 받아 무엇을 돌려줬는지' 를 langfuse 서버로 보내, 나중에 웹 화면에서
# │  흐름을 트리로 펼쳐 볼 수 있게 표시만 해두는 것이다. 지워도 결과는 똑같다.
# └──────────────────────────────────────────────────────────────────────
@observe(name="01_질문확장(Multi-Query)")
def fan_out_queries(question, n=4):
    raw = (SUBQUERY_PROMPT | llm | StrOutputParser()).invoke({"question": question})
    return [q.strip() for q in raw.split("\n") if q.strip()][:n]


# (2) RRF 함수 — 직접 구현
#     @observe 를 붙이면 이 순수 파이썬 함수의 인자(쿼리별 검색결과)와
#     리턴(융합된 문서 3개)이 langfuse 화면에 그대로 남는다. LangChain 이 아니어도 잡힌다.
# ┌─ [관측용 · langfuse] ─ 여기부터 한 줄은 실습 로직이 아니다 ──────────────
# │  이 데코레이터는 함수가 하는 일을 바꾸지 않는다. 함수가 호출될 때
# │  '어떤 인자를 받아 무엇을 돌려줬는지' 를 langfuse 서버로 보내, 나중에 웹 화면에서
# │  흐름을 트리로 펼쳐 볼 수 있게 표시만 해두는 것이다. 지워도 결과는 똑같다.
# └──────────────────────────────────────────────────────────────────────
@observe(name="02_RRF융합")
def reciprocal_rank_fusion(results_per_query, k=60, top_k=3):
    """
    results_per_query : List[List[Document]]  쿼리별 검색 결과(순위 순).
    k                 : RRF smoothing 상수 (관례적으로 60).
    top_k             : 최종 반환할 문서 개수.
    """
    scores = defaultdict(float)
    docs_by_key = {}

    # 쿼리별 결과를 돌면서 문서마다 점수를 누적한다.
    # RRF 의 핵심은 "점수를 더하는 게 아니라 순위(rank)를 더한다" 는 것이다.
    # 각 검색기가 매기는 유사도 점수는 스케일이 제각각이라 그대로 더하면 한 쪽이 판을 먹는데,
    # 순위는 스케일이 없으니 서로 다른 검색 결과를 공정하게 합칠 수 있다.
    for docs in results_per_query:
        for rank, doc in enumerate(docs):      # rank 는 0부터
            key = doc.page_content             # 같은 문서를 여러 쿼리가 찾아오면 본문으로 동일 문서로 본다
            # 1/(k+rank+1) : 1등이면 1/61, 2등이면 1/62 ...
            # k=60 이 크기 때문에 1등과 2등의 차이가 아주 작다 -> "1등 독식"을 막는 완충 장치.
            # 대신 여러 쿼리가 공통으로 뽑아온 문서는 작은 점수가 여러 번 쌓여 위로 올라온다.
            scores[key] += 1.0 / (k + rank + 1)
            docs_by_key[key] = doc

    # 누적 점수가 큰 순서로 정렬해서 상위 top_k 개의 Document 를 돌려준다.
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [docs_by_key[key] for key, _ in ranked[:top_k]]


# (3) 한 번 돌려보기
sub_queries = fan_out_queries(TEST_Q)
print(f"확장 질문 {len(sub_queries)}개:")
for q in sub_queries:
    print(" -", q)

results_per_q = [db.similarity_search(q, k=5) for q in sub_queries]
fused = reciprocal_rank_fusion(results_per_q, k=60, top_k=3)

print("\nRAG-Fusion top-1 문서:")
print(fused[0].page_content[:300] if fused else "(아직 TODO 가 비어 있어 결과가 없습니다)")

# 내가 확인하려고 덧붙인 부분 — 누적 점수를 눈으로 봐야 RRF 가 뭘 한 건지 실감이 난다.
_scores = defaultdict(float)
for _docs in results_per_q:
    for _rank, _doc in enumerate(_docs):
        _scores[_doc.page_content] += 1.0 / (60 + _rank + 1)
print("\n[RRF 점수 상위 5개] 몇 개 쿼리가 공통으로 뽑았는지 보인다")
for _key, _sc in sorted(_scores.items(), key=lambda x: x[1], reverse=True)[:5]:
    _hits = sum(1 for _docs in results_per_q if any(_d.page_content == _key for _d in _docs))
    print(f"  score={_sc:.5f}  {_hits}/{len(results_per_q)}개 쿼리가 검색  | {_key[:60]}...")


확장 질문 4개:
 - 2004년 이명박 서울시장 시절에 대대적으로 개편된 것은 무엇인가요?
 - 이명박이 서울시장으로 있던 2004년에 크게 변화된 것은 무엇인가요?
 - 2004년 서울시장 이명박 재임 중에 전면적으로 바뀐 것은 무엇인가요?
 - 이명박이 2004년 서울시장으로 있을 때 전면 개편한 것은 무엇인가요?



RAG-Fusion top-1 문서:
2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도 했다. 하지만 새 교통체계가 정착되면서 많은 긍정적인 효과를 가져오게 된다. 중앙버스차로 도입으로 버스의 평균 속도가 증가하여 정시에 도착하는 빈도가 늘어났고 환승제도로 인한 교

[RRF 점수 상위 5개] 몇 개 쿼리가 공통으로 뽑았는지 보인다
  score=0.06557  4/4개 쿼리가 검색  | 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 ...
  score=0.06426  4/4개 쿼리가 검색  | 2010년 한나라당 당내경선에서 나경원, 김충환 등의 경쟁자를 물리치고, 민선 5기 지방선거에서 서울시장 재...
  score=0.04663  3/4개 쿼리가 검색  | 사법연수원을 16기로 수료한 후 변호사 생활을 하다가 16대 총선에서 서울 강남을에 출마하여 정계에 입문했다...
  score=0.03175  2/4개 쿼리가 검색  | 2004년 이명박 전 서울시장의 대중교통 정책으로 서울시민은 대중교통 환승시 무료나 할인된 요금을 적용받게 ...
  score=0.03101  2/4개 쿼리가 검색  | 한나라당 소속인 오세훈 서울시장은 소외 계층을 대상으로 선별적인 무상 급식을 시행 중에 있었으나, 야당이 다...


## Step 3 : 패턴 ② HyDE — 가상의 ‘정답’으로 진짜 정답 찾기  

질문은 짧은 의문문, 정답은 긴 평서문이라 둘의 임베딩이 의외로 멀 수 있습니다. **HyDE(Hypothetical Document Embeddings)** 는 검색 전에 LLM에게 ‘가상의 정답’을 쓰게 한 뒤, 그 가상 답변을 임베딩해서 검색합니다.

직접 구현해 보겠습니다.

In [10]:
HYDE_PROMPT = ChatPromptTemplate.from_template(
    "당신은 해당 분야 전문가입니다. 다음 질문에 대해 그럴듯한 한국어 답변 한 문단을 작성하세요. "
    "확실하지 않다면 가장 합리적인 추측을 적어주세요.\n\n"
    "질문: {question}\n\n가상 답변:"
)

hyde_generator = HYDE_PROMPT | llm | StrOutputParser()

# ┌─ [관측용 · langfuse] ─ 여기부터 한 줄은 실습 로직이 아니다 ──────────────
# │  이 데코레이터는 함수가 하는 일을 바꾸지 않는다. 함수가 호출될 때
# │  '어떤 인자를 받아 무엇을 돌려줬는지' 를 langfuse 서버로 보내, 나중에 웹 화면에서
# │  흐름을 트리로 펼쳐 볼 수 있게 표시만 해두는 것이다. 지워도 결과는 똑같다.
# └──────────────────────────────────────────────────────────────────────
@observe(name="03_HyDE검색")
def hyde_retrieve(question, k=3):
    """질문 → 가상의 답변 → 가상 답변을 임베딩해 검색"""
    hypothetical = hyde_generator.invoke({"question": question})
    return db.similarity_search(hypothetical, k=k), hypothetical

docs_hyde, hyp = hyde_retrieve(TEST_Q)
print("가상 답변(HyDE):\n", hyp[:300], "\n---")
print("검색된 문서 수:", len(docs_hyde))
print("첫 문서:", docs_hyde[0].page_content[:200])


가상 답변(HyDE):
 2004년 이명박 서울시장 재직 시절, 그는 서울의 교통 체계를 전면적으로 개선하는 데 주력했습니다. 그 중에서도 가장 주목할 만한 것은 버스 중앙차로제 도입과 대중교통 환승 시스템의 개편입니다. 이명박 시장은 서울의 교통 혼잡 문제를 해결하기 위해 버스와 지하철 간의 환승 체계를 강화하고, 버스 전용 차로를 확대하여 대중교통의 효율성을 높였습니다. 이러한 교통 정책은 시민들의 대중교통 이용을 촉진하고, 서울의 교통 흐름을 개선하는 데 기여했습니다. 이 외에도 청계천 복원 사업을 통해 도심 환경을 개선하고, 시민들에게 휴식 공간을  
---
검색된 문서 수: 3
첫 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


## Step 4 : Post-retrieval 강화 — Cross-Encoder Reranking (multilingual)

검색 결과를 그대로 LLM 에 넘기지 않고, **Cross-encoder reranker** 가 (질문, 문단)을 함께 보면서 진짜 관련도를 다시 점수화합니다. 정밀도가 15~30% 개선되는 게 일반적인 보고입니다.

한국어 문서를 다루고 있으므로 다국어를 지원하는 cross-encoder 를 사용합니다. `BAAI/bge-reranker-v2-m3` 는 한국어를 포함한 100개 이상 언어에서 동작합니다. 처음 실행 시 모델 다운로드(~2GB)가 발생합니다.

In [11]:
from sentence_transformers import CrossEncoder

# 다국어 cross-encoder (한국어 포함)
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

# ┌─ [관측용 · langfuse] ─ 여기부터 한 줄은 실습 로직이 아니다 ──────────────
# │  이 데코레이터는 함수가 하는 일을 바꾸지 않는다. 함수가 호출될 때
# │  '어떤 인자를 받아 무엇을 돌려줬는지' 를 langfuse 서버로 보내, 나중에 웹 화면에서
# │  흐름을 트리로 펼쳐 볼 수 있게 표시만 해두는 것이다. 지워도 결과는 똑같다.
# └──────────────────────────────────────────────────────────────────────
@observe(name="04_리랭킹(CrossEncoder)")
def rerank(query, docs, top_k=3):
    """검색된 docs 를 cross-encoder 로 다시 점수화해 상위 top_k 만 반환"""
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:top_k]]

candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(TEST_Q)
top3 = rerank(TEST_Q, candidates, top_k=3)
print(f"후보 {len(candidates)}개 → Reranker 로 상위 3개 선별")
print("최상위 문서:", top3[0].page_content[:200])


/home/gmw/anaconda3/envs/rag2/lib/python3.12/site-packages/transformers/models/xlm_roberta/modeling_xlm_roberta.py:364: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:360.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


후보 10개 → Reranker 로 상위 3개 선별
최상위 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


## Step 5 : Advanced RAG 체인 조립  

위에서 만든 컴포넌트들을 하나의 체인으로 묶습니다. **‘넓게 검색 → Reranker로 좁히기 → LLM 답변’** 패턴이 가장 흔히 쓰입니다.

In [12]:
# ┌─ [관측용 · langfuse] ─ 여기부터 한 줄은 실습 로직이 아니다 ──────────────
# │  이 데코레이터는 함수가 하는 일을 바꾸지 않는다. 함수가 호출될 때
# │  '어떤 인자를 받아 무엇을 돌려줬는지' 를 langfuse 서버로 보내, 나중에 웹 화면에서
# │  흐름을 트리로 펼쳐 볼 수 있게 표시만 해두는 것이다. 지워도 결과는 똑같다.
# └──────────────────────────────────────────────────────────────────────
@observe(name="05_Advanced_RAG(전체)")
def advanced_rag(question):
    # 1) 후보를 넓게 검색 (k=10)
    candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(question)

    # 2) Cross-encoder 로 진짜 관련도 재정렬 후 상위 3개
    #    이 구간만 따로 as_type="retriever" 로 감싼다. 그러면 langfuse 화면에서 '검색 단계'로 분류돼
    #    어떤 문서가 실제 근거로 들어갔는지 본문까지 남는다.
    #    -> 나중에 "출처 3개를 줬는데 답에 실제로 쓰인 건 1개" 같은 걸 화면에서 대조할 수 있다.
    # ┌─ [관측용 · langfuse] ─ 아래 with 한 줄만 관측용이다 ──────────────
    # │  함수로 안 쪼갠 '코드 구간' 을 하나의 단계로 묶어 기록하는 방법.
    # │  as_type="retriever" 로 지정하면 langfuse 가 이걸 '검색 단계' 로 분류해서
    # │  어떤 문서가 근거로 들어갔는지 본문까지 보기 좋게 보여준다.
    # │  안쪽의 rerank(...) 호출이 실제 실습 코드이고, with 와 span.update 는 기록용이다.
    # └──────────────────────────────────────────────────────────────────
    with lf_client.start_as_current_observation(name="후보10개→상위3개", as_type="retriever") as span:
        top = rerank(question, candidates, top_k=3)
        span.update(input={"질문": question, "후보수": len(candidates)},
                    output=[{"page_content": d.page_content[:200]} for d in top])

    # 3) 프롬프트에 컨텍스트로 주입 → 답변
    context = format_docs(top)
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": context, "question": question})
    return answer, top

ans_adv, ctx_adv = advanced_rag(TEST_Q)
print("Advanced RAG 답변:\n", ans_adv)


Advanced RAG 답변:
 대중교통체계입니다.


## Step 5.5 : Self-RAG — 검색 필요성 판단 + 답변 자가 비평

Day2_1 노트에서 강조한 또 하나의 핵심 패턴, **Self-RAG** 입니다. Self-RAG의 핵심은 **LLM이 검색·답변 과정에 스스로 비평(critique)을 끼워 넣는다**는 점입니다.

이번 셀에서는 공식 Self-RAG 모델을 따로 받지 않고, **세 개의 작은 LLM 프롬프트**로 같은 흐름을 흉내내 봅니다.

1. **Retrieve 결정** — 질문이 들어오면, 외부 검색이 필요한지 LLM이 먼저 판단합니다. (`YES`/`NO` 한 단어)
2. **답변 생성** — `YES` 면 일반 RAG, `NO` 면 검색 없이 LLM 단독 답변.
3. **답변 자가 비평** — 생성된 답변이 컨텍스트에 충분히 근거하는지 LLM이 점검합니다. (`SUPPORTED` / `NOT_SUPPORTED`)
4. **보완 재시도** — `NOT_SUPPORTED` 면 Step 3의 **HyDE** 로 검색 쿼리를 바꿔 한 번 더 시도합니다.

코드 골격은 제공해 두었고, **두 군데 핵심 프롬프트만 여러분이 직접 채워주세요.**

In [13]:
# Self-RAG : retrieve 판단 + 자가 비평 + HyDE 재시도

# (1) 검색 필요성 판단 프롬프트 — 직접 작성
#     핵심은 "한 단어만 뱉게" 만드는 것. 설명을 붙이면 아래 decision.startswith("NO") 판정이 깨진다.
RETRIEVE_DECISION_PROMPT = ChatPromptTemplate.from_template(
    "당신은 질문을 분류하는 판정기입니다.\n"
    "아래 질문에 답하려면 외부 문서(백과사전, 뉴스 기사 같은 자료)를 찾아봐야 하는지 판단하세요.\n"
    "- 특정 인물, 사건, 연도, 수치, 고유명사처럼 자료를 찾아봐야 아는 내용이면 YES\n"
    "- 일반 상식, 단순 계산, 단어의 뜻처럼 자료 없이도 답할 수 있으면 NO\n"
    "다른 설명은 절대 붙이지 말고 오직 YES 또는 NO 한 단어만 출력하세요.\n\n"
    "질문: {question}\n\n판정:"
)

# (2) 답변 자가 비평 프롬프트 — 직접 작성
#     "문서에 있느냐"만 보게 만드는 게 중요하다. 사실 여부(내 상식으로 맞나)를 묻는 게 아니다.
CRITIQUE_PROMPT = ChatPromptTemplate.from_template(
    "당신은 답변을 검사하는 판정기입니다.\n"
    "아래 [답변]의 내용이 [문서]만으로 충분히 뒷받침되는지 판단하세요.\n"
    "당신이 알고 있는 지식은 쓰지 말고, 오직 [문서]에 그 내용이 있는지만 보세요.\n"
    "- 답변의 핵심 내용이 문서에 있으면 SUPPORTED\n"
    "- 문서에 없는 내용을 답변이 지어냈거나 문서와 어긋나면 NOT_SUPPORTED\n"
    "다른 설명은 절대 붙이지 말고 오직 SUPPORTED 또는 NOT_SUPPORTED 한 단어만 출력하세요.\n\n"
    "[문서]\n{context}\n\n[답변]\n{answer}\n\n판정:"
)


# ┌─ [관측용 · langfuse] ─ 여기부터 한 줄은 실습 로직이 아니다 ──────────────
# │  이 데코레이터는 함수가 하는 일을 바꾸지 않는다. 함수가 호출될 때
# │  '어떤 인자를 받아 무엇을 돌려줬는지' 를 langfuse 서버로 보내, 나중에 웹 화면에서
# │  흐름을 트리로 펼쳐 볼 수 있게 표시만 해두는 것이다. 지워도 결과는 똑같다.
# └──────────────────────────────────────────────────────────────────────
@observe(name="10_Self-RAG(판단+비평+재시도)")
def self_rag(question, max_retries=1, verbose=True):
    decision = (RETRIEVE_DECISION_PROMPT | llm | StrOutputParser()).invoke(
        {"question": question}).strip().upper()
    if verbose:
        print(f"[1] Retrieve 필요? -> {decision}")

    if decision.startswith("NO"):
        ans = llm.invoke(question).content
        if verbose:
            print("[2] LLM 단독 답변 사용")
        return ans, []

    docs = db.as_retriever(search_kwargs={"k": 3}).invoke(question)

    for attempt in range(max_retries + 1):
        answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
            {"context": format_docs(docs), "question": question})
        critique = (CRITIQUE_PROMPT | llm | StrOutputParser()).invoke(
            {"context": format_docs(docs), "answer": answer}).strip().upper()
        if verbose:
            print(f"[3] 시도 {attempt+1} — 자가 비평: {critique}")

        if "NOT" not in critique:
            return answer, docs

        if attempt < max_retries:
            hyp = hyde_generator.invoke({"question": question})
            docs = db.similarity_search(hyp, k=3)
            if verbose:
                print("[4] NOT_SUPPORTED -> HyDE 가상 답변으로 재검색")

    return answer, docs


ans_sr, ctx_sr = self_rag(TEST_Q)
print("\n=== Self-RAG 최종 답변 ===")
print(ans_sr)

# 검색이 필요 없는 질문도 넣어봐야 (1)번 프롬프트가 진짜 일하는지 알 수 있다.
# 이걸 안 해보면 "YES 만 뱉는 프롬프트"여도 눈치를 못 챈다 — 어제 배운 no-op 함정과 같은 이야기.
print("\n=== 검색이 필요 없는 질문으로 대조 실험 ===")
ans_sr2, ctx_sr2 = self_rag("3 곱하기 7은 얼마야?")
print(ans_sr2)


[1] Retrieve 필요? -> YES


[3] 시도 1 — 자가 비평: NOT_SUPPORTED


[4] NOT_SUPPORTED -> HyDE 가상 답변으로 재검색


[3] 시도 2 — 자가 비평: NOT_SUPPORTED

=== Self-RAG 최종 답변 ===
대중교통체계입니다.

=== 검색이 필요 없는 질문으로 대조 실험 ===


[1] Retrieve 필요? -> NO


[2] LLM 단독 답변 사용
3 곱하기 7은 21입니다.


## Step 6 : RAGAS 평가용 데이터셋 만들기

RAGAS 는 네 가지 자료가 필요합니다.
- `user_input` — 사용자 질문
- `response`   — RAG 가 생성한 답변
- `retrieved_contexts` — RAG 가 참고한 문서들
- `reference`  — 모범 답안 (Ground Truth)

**KorQuAD 는 사람이 작성한 정답이 데이터셋에 이미 포함**되어 있어, `reference` 를 따로 작성할 필요 없이 그대로 가져다 씁니다. 같은 질문 셋을 **Naive RAG** 와 **Advanced RAG** 두 가지로 풀고 결과를 비교합니다.

토큰 비용 통제를 위해 평가 질문은 5개만 사용합니다. (늘리려면 `EVAL_N` 변경)

In [14]:
# 평가용 질문/정답 자동 추출 (KorQuAD)
EVAL_N = 20  # 평가에 쓸 질문 수. 늘리면 표본 흔들림은 줄지만 OpenAI 토큰 비용이 그만큼 는다.
eval_samples = list(raw_ds)[:EVAL_N]
questions = [ex["question"] for ex in eval_samples]
ground_truths = [ex["answers"]["text"][0] for ex in eval_samples]

# Naive RAG 로 답변 + 컨텍스트 수집
# 여기서 하는 일: 질문 20개를 두 파이프라인에 각각 통과시켜
#   (1) 답변  (2) 그 답을 만들 때 근거로 준 문서들
# 두 가지를 모아둔다. RAGAS 는 이 둘과 정답(ground truth)을 비교해 채점한다.
# ★Naive 와 Advanced 에 '똑같은 질문 20개' 를 주는 게 중요하다. 질문이 다르면 비교가 성립하지 않는다.
naive_answers, naive_contexts = [], []
for q in questions:
    ctx = naive_retriever.invoke(q)
    a = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(ctx), "question": q})
    naive_answers.append(a)
    naive_contexts.append([d.page_content for d in ctx])

# Advanced RAG 로 답변 + 컨텍스트 수집
adv_answers, adv_contexts = [], []
for q in questions:
    a, ctx = advanced_rag(q)
    adv_answers.append(a)
    adv_contexts.append([d.page_content for d in ctx])

print(f"데이터셋 준비 완료 — {EVAL_N}개 질문 × 2개 파이프라인")

데이터셋 준비 완료 — 20개 질문 × 2개 파이프라인


In [15]:
from datasets import Dataset

# RAGAS 가 요구하는 4개 열 이름에 맞춰 데이터셋을 만든다. 이름이 하나라도 다르면 채점이 안 된다.
#   user_input=질문 / response=우리 파이프라인의 답 / retrieved_contexts=근거 문서들 / reference=정답
def make_dataset(answers, contexts):
    return Dataset.from_dict({
        "user_input":         questions,
        "response":           answers,
        "retrieved_contexts": contexts,
        "reference":          ground_truths,
    })

naive_ds = make_dataset(naive_answers, naive_contexts)
adv_ds   = make_dataset(adv_answers,   adv_contexts)

## Step 7 : RAGAS로 4대 지표 계산하기  

Judge LLM은 `gpt-4o-mini`로, 임베딩은 `text-embedding-3-small`로 설정합니다.  
(Judge에 더 강한 모델을 쓰면 채점은 더 정교해지지만 비용이 늘어납니다.)

In [16]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness, answer_relevancy,
    context_precision, context_recall,
)

judge_llm  = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_emb  = OpenAIEmbeddings(model="text-embedding-3-small")
metrics    = [faithfulness, answer_relevancy,
              context_precision, context_recall]

# ─── [내가 추가한 패치] answer_relevancy 한국어 대응 ────────────────────────
# 어제 이 지표가 멀쩡한 한국어 답변에 -0.0166 (음수!) 을 주는 걸 발견했다.
# 원인: answer_relevancy 는 "답변만 보고 질문을 거꾸로 만들어" 원래 질문과 코사인 유사도를 잰다.
#       그런데 ragas 0.2.10 의 역질문 프롬프트는 few-shot 예시가 전부 영어라, 한국어 답변을 주면
#       영어 질문을 만들어낸다. 한국어 질문 vs 영어 역질문 -> 유사도가 바닥(음수까지) 이 된다.
#       ragas 의 언어 적응 기능(adapt_prompts)에는 'korean' 자체가 없어서 쓸 수 없었다.
# 처방: 역질문 프롬프트의 지시문에 "응답과 같은 언어로 쓰라" 한 문장을 직접 못 박는다.
#       실측으로 0.232 -> 0.928 로 올라가는 걸 확인했다.
# 이 패치 없이 비교표를 뽑으면 answer_relevancy 열 전체가 '언어 불일치 노이즈' 라서 읽으나 마나다.
_ar_prompts = answer_relevancy.get_prompts()
for _name, _p in _ar_prompts.items():
    if "동일한 언어" not in _p.instruction:      # 두 번 실행돼도 중복으로 안 붙게
        _p.instruction = _p.instruction + \
            " 생성하는 question 은 반드시 response 와 동일한 언어(한국어)로 작성하세요."
answer_relevancy.set_prompts(**_ar_prompts)
print("[패치] answer_relevancy 역질문 프롬프트에 '응답과 같은 언어' 지시 추가함")
# ───────────────────────────────────────────────────────────────────────

print("=== Naive RAG 채점 ===")
naive_result = evaluate(naive_ds, metrics=metrics,
                        llm=judge_llm, embeddings=judge_emb,
                        raise_exceptions=False)

print("=== Advanced RAG 채점 ===")
adv_result = evaluate(adv_ds, metrics=metrics,
                      llm=judge_llm, embeddings=judge_emb,
                      raise_exceptions=False)


[패치] answer_relevancy 역질문 프롬프트에 '응답과 같은 언어' 지시 추가함
=== Naive RAG 채점 ===


=== Advanced RAG 채점 ===


In [17]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

naive_df = naive_result.to_pandas()
adv_df   = adv_result.to_pandas()

# 문항별 점수표(20행)를 지표별 평균 한 줄로 접는다. 이 함수를 KLUE 쪽에서도 그대로 재사용한다.
def summary(df, label):
    cols = ["faithfulness", "answer_relevancy",
            "context_precision", "context_recall"]
    avg = df[cols].mean()
    avg.name = label
    return avg

compare = pd.concat([summary(naive_df, "Naive RAG"),
                     summary(adv_df,   "Advanced RAG")], axis=1)
print(compare.round(3))
print("\nDelta (Advanced - Naive):")
print((compare["Advanced RAG"] - compare["Naive RAG"]).round(3))

                   Naive RAG  Advanced RAG
faithfulness           0.675         0.850
answer_relevancy       0.208         0.184
context_precision      0.692         0.892
context_recall         0.750         0.900

Delta (Advanced - Naive):
faithfulness         0.175
answer_relevancy    -0.024
context_precision    0.200
context_recall       0.150
dtype: float64


> ### 실험 기록 2 — 같은 코드를 세 번 돌렸더니 숫자가 매번 달라졌다
>
> 위 사고(실험 기록 1) 때문에 노트북을 처음부터 다시 돌리게 됐는데, 뜻밖의 것을 보게 됐다.
> **KorQuAD 부분은 코드도 데이터도 seed 도 하나도 안 바꿨는데 점수가 달라졌다.**
>
> | 지표 | 1차 실행 | 2차 실행 | 3차 실행(위 표 = 제출본) |
> |---|---|---|---|
> | faithfulness | 0.650 -> 0.800 (+0.150) | 0.650 -> 0.850 (+0.200) | 0.675 -> 0.850 (+0.175) |
> | answer_relevancy | 0.206 -> 0.181 (-0.025) | 0.271 -> 0.174 (-0.097) | 0.208 -> 0.184 (-0.024) |
> | context_precision | 0.692 -> 0.892 (+0.200) | 0.692 -> 0.883 (+0.192) | 0.692 -> 0.892 (+0.200) |
> | context_recall | 0.750 -> 0.900 (+0.150) | 0.750 -> 0.900 (+0.150) | 0.750 -> 0.900 (+0.150) |
>
> (2차는 Vector DB 사고를 고치느라, 3차는 Langfuse 관측 코드를 넣느라 다시 돌린 것이다.
>  결과적으로 같은 코드를 세 번 돌린 셈이 됐다.)
>
> **왜 달라지나**: 답변을 만드는 LLM 과 채점하는 LLM 을 둘 다 `temperature=0` 으로 뒀지만,
> 그게 "완전히 똑같은 출력" 을 보장하지는 않는다. 특히 RAGAS 는 채점 자체를 LLM 에게 맡기는
> LLM-as-Judge 방식이라, 심판이 조금만 다르게 판단해도 20문항 평균이 흔들린다.
>
> **이게 왜 중요하냐면**, 개선폭 1위가 실행마다 바뀌었다.
> 1차는 context_precision(+0.200)이 1위, 2차는 faithfulness(+0.200)가 1위,
> 3차는 다시 context_precision(+0.200)이 1위다. **한 번만 돌리고 "1위는 이것" 이라고 쓰면 틀린다.**
>
> 반대로 세 번 내내 흔들리지 않은 것도 있다. `context_precision` 은 세 번 다 +0.19~+0.20 이었고,
> `context_recall` 은 세 번 다 정확히 +0.150 이었다. **흔들리는 지표와 안 흔들리는 지표가 갈린다**는 것도
> 세 번 돌려보고서야 알았다.
>
> 교재의 해석 가이드에도 "20문항에서 ±0.05 는 표본 noise" 라고 적혀 있는데,
> 나는 그 문장을 읽기만 한 게 아니라 **내 실행 세 개로 직접 확인하게 됐다.**
> 그래서 이 노트북의 결론은 전부 "이 정도 크기의 차이는 우연일 수 있다" 를 깔고 읽어야 한다.
> Step K 의 paired t-test 를 넣은 이유가 이것이다.


### 결과 해석 가이드

위 비교표를 처음 보면 **‘Advanced 가 더 나쁜 거 아닌가?’** 라는 착각을 하기 쉽습니다. KorQuAD 위에서의 결과 해석 방법을 정리합니다.

**1. `context_precision` 의 개선 (+) 이 Advanced RAG 의 핵심 효과**
검색 결과의 ‘상단’에 정답 문단을 두는 일을 Reranker 가 잘 했다는 의미. Δ가 0.05~0.15 정도면 잘 작동.

**2. `context_recall = 1.0` 으로 포화될 수 있다**
unique context 가 800개 정도면 Naive top-3 에도 정답이 거의 항상 들어옵니다. 이 지표는 더 큰 DB(수만 문서)에서 차이가 드러납니다.

**3. `faithfulness` 가 살짝 떨어질 수 있다**
Reranker 가 컨텍스트를 ‘짧고 집중’ 시키면 LLM이 그 좁은 정보에서 답을 만들 때 일부 주장이 “미뒷받침” 으로 채점되어 점수가 약간 내려갈 수 있음. **정상 범위 (-0.1 이내)**.

**4. `answer_relevancy` 가 0.2~0.4 로 낮은 이유 — KorQuAD 의 구조적 특성**
KorQuAD 정답은 *‘대중교통체계’* 같이 한 단어~한 구절. RAG 답변도 짧게 나오는데, RAGAS 의 `answer_relevancy` 는 **답변에서 질문을 역추론**해 원래 질문과의 유사도를 계산합니다. 답변이 한 단어면 역추론이 흐려져 점수가 낮아집니다. **모델 잘못이 아닌 데이터셋 특성**.

**5. 표본 20개로도 Δ가 ±0.05 이내면 ‘차이 없음’으로 봐야 한다**
20문항에서 ±0.05 는 표본 noise. 더 확실한 판단이 필요하면 `scipy.stats.ttest_rel` 로 통계 검정을 하거나 50~100문항으로 늘려야 합니다.

**6. 한국어 짧은 정답 벤치마크의 한계**
KorQuAD/KLUE-MRC 처럼 정답이 짧은 extractive QA 벤치마크는 `context_precision` 위주로 평가 효과를 봐야 하고, `answer_relevancy` 는 절대값보다 **Naive 대비 상대 변화**로 읽어야 합니다.

### Quiz  
위 표에서 Advanced RAG가 가장 크게 개선한 지표는 무엇인가요? 그리고 그 지표는 우리가 적용한 **어떤 기법**과 가장 직접적으로 연결될까요?  

**Answer (예시)**:  
보통 `context_precision`이 가장 크게 오릅니다. 이는 우리가 추가한 **Reranker**가 ‘진짜 관련도가 높은 문서를 상위에 두는 일’을 잘 했다는 의미입니다.  `context_recall`은 **Multi-Query**가 검색 폭을 넓혔다면 같이 오릅니다.  `faithfulness`와 `answer_relevancy`는 컨텍스트 품질이 올라가면 부수적으로 개선됩니다.

### Quiz — 내가 적은 답

> 위 표에서 Advanced RAG가 가장 크게 개선한 지표는 무엇인가요? 그 지표는 어떤 기법과 가장 직접적으로 연결될까요?

**답: `context_precision` 이고, 연결되는 기법은 Cross-Encoder Reranker 다.**
다만 실행할 때마다 순위가 흔들려서, 단정하지 않고 아래처럼 적는다.

이번 실행(제출본) 기준 개선폭은 이랬다.

| 지표 | Naive | Advanced | 차이 | t-test |
|---|---|---|---|---|
| context_precision | 0.692 | 0.892 | **+0.200** | p=0.0215 **유의미** |
| faithfulness | 0.675 | 0.850 | **+0.175** | p=0.0493 **유의미** |
| context_recall | 0.750 | 0.900 | +0.150 | p=0.0828 차이 없음 |
| answer_relevancy | 0.208 | 0.184 | -0.024 | p=0.4919 차이 없음 |

다만 "1등" 을 단정하지는 않는다. 위 '실험 기록 2' 에 적었듯 같은 코드를 세 번 돌렸더니
1위가 context_precision -> faithfulness -> context_precision 으로 왔다 갔다 했다.
**둘 다 크게 올랐고, 그중 context_precision 이 세 번 내내 +0.19~+0.20 으로 가장 안정적이었다** 가
정확한 표현이다.

**기법과의 연결은 이렇게 본다.**

- `context_precision` <- **Reranker (직접)**.
  이 지표는 "가져온 문서 중 쓸모 있는 것이 얼마나 앞쪽에 있느냐" 를 본다.
  임베딩은 질문과 문서를 따로 벡터로 만들어 거리만 재서 "비슷해 보이는" 문서를 위에 올리는 실수를 한다.
  리랭커는 (질문, 문서) 를 한 쌍으로 넣어 관련도를 직접 매기니, 넓게 뽑은 10개를 다시 줄 세워
  진짜 쓸모 있는 3개를 위로 올린다. 지표의 정의와 기법이 그대로 맞물린다.
- `faithfulness` <- **Reranker (간접)**.
  리랭커가 답변을 고친 게 아니라, **답변할 재료를 좋게 줬을 뿐**이다.
  컨텍스트가 정확해지니 LLM 이 문서에 없는 말을 덜 지어냈다.
- `context_recall` <- **후보를 k=3 이 아니라 k=10 으로 넓게 뽑은 것**.
  정답 문서가 아예 후보에 들어오느냐를 보는 지표라, 폭을 넓힌 게 여기에 기여한다.
- `answer_relevancy` 는 유일하게 떨어졌는데, **파이프라인이 나빠진 게 아니다.**
  KorQuAD 정답이 평균 6.0자짜리 한 단어라 이 지표 자체가 제대로 작동하지 않는 구간이다
  (자세한 건 회고 3-(1) 에 정리했다). 세 번의 실행에서 -0.025 / -0.097 / -0.024 로 편차도 제일 컸다.


## Step 8 : (선택) 평가 데이터를 LLM으로 자동 생성하기  

현업에서는 모범 답안(`reference`)을 사람이 직접 작성하는 게 가장 큰 부담입니다.  
RAGAS는 **원본 문서만 주면 Question·Reference·Context 한 세트를 자동으로 만들어 주는** 기능을 제공합니다.  
자세한 사용법은 공식 문서를 참고하세요.

https://docs.ragas.io/en/stable/getstarted/rag_testset_generation/

---
# 추가 실습 — KLUE-MRC 한국어 뉴스 MRC 벤치마크로 RAG 평가하기

메인 실습은 위키 기반 **KorQuAD v1** 으로 진행했습니다. 이번 추가 실습은 도메인을 바꿔, **한국어 뉴스 기사 기반의 MRC 벤치마크 KLUE-MRC** 위에서 같은 파이프라인을 처음부터 다시 조립해 봅니다.

**KLUE-MRC**
- 카카오·네이버 등 한국 NLP 팀이 함께 만든 한국어 표준 벤치마크 KLUE 의 MRC 태스크
- 한국어 **뉴스 기사** 기반 (KorQuAD 의 위키와 도메인이 다름)
- 사람이 직접 작성한 정답 포함
- **`is_impossible=True`** 인 답할 수 없는 질문도 일부 포함 → 데이터 필터링이 필요한 도전적 케이스

위키 기반 KorQuAD 와 비교했을 때 어떤 차이(질문 스타일, 검색 난이도, 점수 분포)가 나는지 직접 관찰해 보세요.

이번에도 일부만 샘플링해서 토큰 비용을 통제합니다.
- Vector DB 에 들어갈 unique context: 약 200개
- 평가 질문: 20개
- 예상 비용: GPT-4o-mini 기준 RAGAS 평가까지 합쳐서 약 \$0.10 ~ \$0.20

**📥 데이터셋 다운로드 / 출처**
- HuggingFace `datasets` 자동 다운로드: <https://huggingface.co/datasets/klue>
- KLUE 공식 사이트: <https://klue-benchmark.com/>
- KLUE 논문: <https://arxiv.org/abs/2105.09680>

> 다른 데이터셋으로 한 번 더 해보고 싶다면:  
> - MIRACL 한국어: <https://huggingface.co/datasets/miracl/miracl> (config: `ko`)  
> - 영어 SQuAD: <https://huggingface.co/datasets/rajpurkar/squad>

### Step A. 데이터셋 로드

`datasets` 라이브러리로 KLUE-MRC 를 한 줄에 받아옵니다. KLUE 는 여러 sub-task 가 있는 멀티태스크 벤치마크라서 config 이름 `"mrc"` 를 명시해야 합니다.

데이터셋 페이지: <https://huggingface.co/datasets/klue>

In [18]:
from datasets import load_dataset

ds_klue = load_dataset("klue", "mrc", split="validation")
print(ds_klue)
print("\n--- 샘플 1건 ---")
print({k: ds_klue[0][k] for k in ds_klue.column_names})

Dataset({
    features: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers'],
    num_rows: 5841
})

--- 샘플 1건 ---
{'title': 'BMW 코리아, 창립 25주년 기념 ‘BMW 코리아 25주년 에디션’ 한정 출시', 'context': 'BMW 코리아(대표 한상윤)는 창립 25주년을 기념하는 ‘BMW 코리아 25주년 에디션’을 한정 출시한다고 밝혔다. 이번 BMW 코리아 25주년 에디션(이하 25주년 에디션)은 BMW 3시리즈와 5시리즈, 7시리즈, 8시리즈 총 4종, 6개 모델로 출시되며, BMW 클래식 모델들로 선보인 바 있는 헤리티지 컬러가 차체에 적용돼 레트로한 느낌과 신구의 조화가 어우러진 차별화된 매력을 자랑한다. 먼저 뉴 320i 및 뉴 320d 25주년 에디션은 트림에 따라 옥스포드 그린(50대 한정) 또는 마카오 블루(50대 한정) 컬러가 적용된다. 럭셔리 라인에 적용되는 옥스포드 그린은 지난 1999년 3세대 3시리즈를 통해 처음 선보인 색상으로 짙은 녹색과 풍부한 펄이 오묘한 조화를 이루는 것이 특징이다. M 스포츠 패키지 트림에 적용되는 마카오 블루는 1988년 2세대 3시리즈를 통해 처음 선보인 바 있으며, 보랏빛 감도는 컬러감이 매력이다. 뉴 520d 25주년 에디션(25대 한정)은 프로즌 브릴리언트 화이트 컬러로 출시된다. BMW가 2011년에 처음 선보인 프로즌 브릴리언트 화이트는 한층 더 환하고 깊은 색감을 자랑하며, 특히 표면을 무광으로 마감해 특별함을 더했다. 뉴 530i 25주년 에디션(25대 한정)은 뉴 3시리즈 25주년 에디션에도 적용된 마카오 블루 컬러가 조합된다. 뉴 740Li 25주년 에디션(7대 한정)에는 말라카이트 그린 다크 색상이 적용된다. 잔잔하면서도 오묘한 깊은 녹색을 발산하는 말라카이트 그린 다크는 장식재로 

### Step B. Context 추출 + 중복 제거 (+ is_impossible 필터링)

KLUE-MRC 에는 KorQuAD 에는 없는 **`is_impossible=True`** 케이스가 섞여 있습니다 (= context 만 보고는 답할 수 없는 질문). 평가용 ground_truth 가 비어 있으면 RAGAS 의 `context_recall` 이 깨지므로, 답이 있는 샘플만 남기세요.

- `ds_klue.filter(lambda x: not x["is_impossible"])` 로 답 있는 것만 추리고
- `shuffle(seed=42).select(range(300))` 으로 300개 샘플링
- 그 중 `context` 필드 기준으로 중복 제거 (보통 150~200개)
- 각각을 `Document(page_content=..., metadata={"title": ex["title"]})` 로 감싸 `context_docs` 에 담기

In [19]:
# Step B. 답할 수 있는 것만 남기고, context 중복 제거해서 Document 로 감싸기

import random
from langchain_core.documents import Document

# (1) is_impossible=True 인 샘플 버리기.
#     이건 "지문을 봐도 답이 없는" 질문이라 answers["text"] 가 비어 있다.
#     그대로 두면 ground_truth 가 빈 문자열이 되어 RAGAS 의 context_recall 채점이 깨진다.
ds_klue_answerable = ds_klue.filter(lambda x: not x["is_impossible"])
print(f"전체 {len(ds_klue)}건 -> 답이 있는 것만 {len(ds_klue_answerable)}건 "
      f"(버린 것 {len(ds_klue) - len(ds_klue_answerable)}건)")

# (2) 300개 샘플링 (seed 고정 — 다시 돌려도 같은 데이터가 나오게)
ds_klue_sample = ds_klue_answerable.shuffle(seed=42).select(range(300))

# (3) context 중복 제거. 하나의 기사에 여러 질문이 달려 있어서 그냥 넣으면 같은 기사가 여러 번 들어간다.
unique_klue = {}
for ex in ds_klue_sample:
    if ex["context"] not in unique_klue:
        unique_klue[ex["context"]] = ex["title"]

# ★ 변수 이름을 context_docs 가 아니라 context_docs_klue 로 둔다.
#   메인 실습(KorQuAD)에서 만든 context_docs 를 덮어쓰면 두 도메인 비교가 불가능해진다.
#   아래 _klue 접미사도 전부 같은 이유다.
context_docs_klue = [Document(page_content=c, metadata={"title": t})
                     for c, t in unique_klue.items()]

print(f"샘플 300건 -> unique context {len(context_docs_klue)}개 "
      f"(기사 1개에 질문이 여러 개라 이만큼 줄었다)")
print("\n--- 첫 문서 미리보기 ---")
print("title:", context_docs_klue[0].metadata["title"])
print(context_docs_klue[0].page_content[:200])

# 뉴스 기사가 위키보다 얼마나 긴지 숫자로 봐두면 뒤에서 점수 해석할 때 쓸모가 있다.
_len_klue = [tiktoken_len(d.page_content) for d in context_docs_klue]
print(f"\n[문서 길이] KLUE 뉴스 평균 {sum(_len_klue)/len(_len_klue):.0f} 토큰 "
      f"(최소 {min(_len_klue)}, 최대 {max(_len_klue)})")


전체 5841건 -> 답이 있는 것만 4008건 (버린 것 1833건)
샘플 300건 -> unique context 299개 (기사 1개에 질문이 여러 개라 이만큼 줄었다)

--- 첫 문서 미리보기 ---
title: 또 해킹당한 가상화폐
국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20

[문서 길이] KLUE 뉴스 평균 1063 토큰 (최소 474, 최대 2123)


### Step C. Embedding + VectorStore

메인 실습에서 만든 `embedding` (`OpenAIEmbeddings(model="text-embedding-3-small")`) 을 그대로 재사용해, `context_docs` 로 새 Chroma DB `db_klue` 를 만드세요. (메인 실습의 `db` 변수를 덮어쓰지 마세요. 비교가 안 됩니다.)

> ⚠️ **batch 적재 필수** — KLUE-MRC 의 뉴스 context 는 평균 토큰 수가 커서, 150개 이상을 한 번에 `Chroma.from_documents` 로 넘기면 OpenAI embeddings 의 **300k 토큰/요청 한도** 에 걸려 `BadRequestError` 가 납니다. 메인 cell 8 처럼 100개씩 batch 로 `add_documents` 호출하세요:
> ```python
> db_klue = Chroma(embedding_function=embedding)
> BATCH = 100
> for i in range(0, len(context_docs), BATCH):
>     db_klue.add_documents(context_docs[i:i+BATCH])
> ```

> 인덱싱 토큰 비용: 약 200개 context × 평균 600 토큰 ≈ **120k 토큰** (≈ \$0.003)

In [20]:
# Step C. Embedding + VectorStore (db_klue)

# 임베딩 모델은 메인 실습과 같은 것을 재사용한다. 모델을 바꾸면 도메인 차이인지
# 임베딩 차이인지 구분할 수 없게 된다 — 비교 실험에서는 한 번에 하나만 바꿔야 한다.

# ★★ collection_name 을 반드시 따로 준다 ★★
# 처음에 그냥 Chroma(embedding_function=embedding) 로 만들었더니, 메인 실습의 db 와
# **같은 컬렉션**을 가리켰다. langchain 의 Chroma 는 컬렉션 이름을 안 주면 기본값 "langchain" 을 쓰는데,
# 메모리 클라이언트가 그 이름을 공유하기 때문이다.
# 그 결과 db_klue 에 KorQuAD 위키 청크 1264개가 이미 들어 있는 상태에서 뉴스 299개가 얹혀서,
# 뉴스 질문을 던져도 위키 문서가 후보로 딸려 나왔다 (= 도메인 비교가 오염).
# 에러는 한 줄도 안 났다. 아래 개수 출력을 안 찍었으면 못 잡았을 사고다.
db_klue = Chroma(embedding_function=embedding, collection_name="klue_mrc")

# 100개씩 잘라 넣는 이유: 뉴스 기사는 위키보다 길어서, 200개를 한 번에 임베딩 요청하면
# OpenAI embeddings 의 300k 토큰/요청 한도에 걸려 BadRequestError 가 난다.
BATCH = 100
for i in range(0, len(context_docs_klue), BATCH):
    db_klue.add_documents(context_docs_klue[i:i+BATCH])
    print(f"  적재 {min(i+BATCH, len(context_docs_klue))}/{len(context_docs_klue)}")

# ★ 두 DB 가 진짜 별개인지 확인한다. 개수가 같으면 같은 컬렉션을 보고 있다는 신호다.
print(f"\ndb_klue      : 컬렉션 '{db_klue._collection.name}' — 문서 {db_klue._collection.count()}개 "
      f"(뉴스 {len(context_docs_klue)}개가 들어가야 맞다)")
print(f"메인 db      : 컬렉션 '{db._collection.name}' — 문서 {db._collection.count()}개 "
      f"(위키 청크 {len(docs)}개 그대로여야 맞다)")
assert db_klue._collection.name != db._collection.name, "두 DB 가 같은 컬렉션을 쓰고 있다!"
assert db_klue._collection.count() == len(context_docs_klue), "db_klue 에 다른 문서가 섞였다!"
print("확인 완료 — 두 DB 는 서로 다른 컬렉션이고, 섞이지 않았다.")


  적재 100/299


  적재 200/299


  적재 299/299

db_klue      : 컬렉션 'klue_mrc' — 문서 299개 (뉴스 299개가 들어가야 맞다)
메인 db      : 컬렉션 'langchain' — 문서 1264개 (위키 청크 1264개 그대로여야 맞다)
확인 완료 — 두 DB 는 서로 다른 컬렉션이고, 섞이지 않았다.


> ### 실험 기록 1 — 여기서 사고가 났었다: 두 Vector DB 가 사실은 하나였다
>
> 위 Step C 를 처음엔 안내대로 이렇게 썼다.
>
> ```python
> db_klue = Chroma(embedding_function=embedding)   # collection_name 을 안 줬다
> ```
>
> 에러는 없었다. 적재도 됐고 검색도 됐고 뒤의 RAGAS 표까지 멀쩡한 숫자로 나왔다.
> 그런데 확인용으로 찍어둔 문서 개수가 이렇게 나왔다.
>
> ```
> db_klue 적재 완료 — 문서 1563개
> 메인 db(KorQuAD) 는 그대로 — 문서 1563개
> ```
>
> **두 DB 의 문서 수가 똑같다.** 계산해보니 KorQuAD 청크 1264개 + KLUE 뉴스 299개 = 정확히 1563 이었다.
> 즉 두 번째 DB 에 뉴스만 들어간 게 아니라, **위키 문서가 이미 들어 있는 곳에 뉴스를 얹은 것**이다.
>
> **원인**: langchain 의 `Chroma` 는 `collection_name` 을 안 주면 기본값 `"langchain"` 을 쓴다.
> 메모리 클라이언트가 그 이름을 공유하기 때문에, 변수는 `db` 와 `db_klue` 로 두 개였지만
> 실제 저장소는 하나였다.
>
> **무엇이 망가졌나**: 뉴스 질문을 던져도 위키 문서가 후보로 딸려 나온다.
> "도메인을 바꾸면 무엇이 달라지나" 를 보려던 이번 추가 실습의 목적 자체가 오염된다.
> 그런데도 표는 정상으로 보였다 — 이게 제일 무서운 부분이다.
>
> **고친 방법**
>
> ```python
> db_klue = Chroma(embedding_function=embedding, collection_name="klue_mrc")   # 이름을 따로 준다
> ```
>
> 그리고 다시는 조용히 지나가지 않도록 `assert` 두 줄을 박았다.
> (컬렉션 이름이 서로 다른지 / db_klue 문서 수가 뉴스 개수와 정확히 같은지)
> 고친 뒤 다시 돌린 출력이 위 셀에 있는 이것이다.
>
> ```
> db_klue      : 컬렉션 'klue_mrc' — 문서 299개
> 메인 db      : 컬렉션 'langchain' — 문서 1264개
> 확인 완료 — 두 DB 는 서로 다른 컬렉션이고, 섞이지 않았다.
> ```
>
> 이 사고 때문에 노트북을 처음부터 한 번 더 돌렸다. 그 재실행이 아래 '실험 기록 2' 를 낳았다.


### Step D. 평가용 질문/정답 세트 추출

Step B 에서 필터링·샘플링한 데이터 중 **앞에서 20개**를 평가용으로 떼어내세요.

- `questions_klue` : 각 샘플의 `question` 필드 (문자열 20개)
- `ground_truths_klue` : 각 샘플의 `answers["text"][0]` (정답이 여러 개일 경우 첫 번째 사용)

> 참고: KLUE-MRC 는 정답이 한 구절~한 문장 단위의 **extractive QA** 입니다. 짧은 정답은 RAGAS 의 `context_recall` 변동성을 키우는 경향이 있으니, 평균을 함께 봐주세요.

In [21]:
# Step D. 평가용 질문/정답 20개 뽑기

# Step B 에서 필터링·샘플링한 ds_klue_sample 의 앞 20개를 쓴다.
# (context 중복 제거 전 원본 순서 기준이라, 같은 기사에 대한 질문이 겹칠 수 있지만
#  그 기사들은 전부 db_klue 안에 들어 있으므로 검색은 정상적으로 된다.)
eval_samples_klue = list(ds_klue_sample)[:20]
questions_klue     = [ex["question"] for ex in eval_samples_klue]
ground_truths_klue = [ex["answers"]["text"][0] for ex in eval_samples_klue]

print(f"평가 질문 {len(questions_klue)}개 / 정답 {len(ground_truths_klue)}개")
print("\n--- 앞 3개 ---")
for q, g in list(zip(questions_klue, ground_truths_klue))[:3]:
    print(f"Q: {q}\nA: {g}\n")

# 정답 길이를 재본다. KorQuAD 와 비교하면 뒤에서 점수 차이를 설명할 때 근거가 된다.
_gt_len = [len(g) for g in ground_truths_klue]
print(f"[정답 길이] KLUE 평균 {sum(_gt_len)/len(_gt_len):.1f}자 "
      f"(최소 {min(_gt_len)}, 최대 {max(_gt_len)})")
_gt_len_kq = [len(g) for g in ground_truths]
print(f"[정답 길이] KorQuAD 평균 {sum(_gt_len_kq)/len(_gt_len_kq):.1f}자 "
      f"(최소 {min(_gt_len_kq)}, 최대 {max(_gt_len_kq)})")


평가 질문 20개 / 정답 20개

--- 앞 3개 ---
Q: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?
A: 두 개

Q: 정유공장 공사는 어느 도시에서 진행되는가?
A: 카르발라

Q: 가장 먼저 리그 진출 팀이 결정되는 경기의 시작 시간은 언제인가?
A: 오후 5시 45분

[정답 길이] KLUE 평균 6.2자 (최소 2, 최대 14)
[정답 길이] KorQuAD 평균 6.0자 (최소 2, 최대 11)


### Step E. Naive RAG 베이스라인 (KLUE)

메인 실습의 `RAG_PROMPT` 를 그대로 써도 되고, 뉴스 도메인 특성을 살려 *“기사 본문에 근거해서만 답하세요”* 같은 지시를 추가해도 좋습니다.

- `naive_retriever_klue = db_klue.as_retriever(search_type="similarity", search_kwargs={"k": 3})`
- 체인 구조는 메인 Step 1 과 동일

In [22]:
# Step E. Naive RAG 베이스라인 (KLUE)

# 프롬프트는 메인 실습의 RAG_PROMPT 를 그대로 쓴다.
# 뉴스용으로 문구를 바꿀 수도 있지만, 그러면 나중에 점수가 달라졌을 때
# "도메인이 달라서인지, 프롬프트를 바꿔서인지" 를 구분할 수 없다. 변수는 하나만 바꾼다.
naive_retriever_klue = db_klue.as_retriever(search_type="similarity",
                                            search_kwargs={"k": 3})

naive_chain_klue = (
    {"context": naive_retriever_klue | format_docs,
     "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

TEST_Q_KLUE = questions_klue[0]
print("Q:", TEST_Q_KLUE)
print("정답(ground truth):", ground_truths_klue[0])
print("A:", naive_chain_klue.invoke(TEST_Q_KLUE))


Q: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?
정답(ground truth): 두 개


A: 두 개의 계좌입니다.


### Step F. Multi-Query Retrieval

메인 Step 2 와 동일하게 `MultiQueryRetriever.from_llm(...)` 으로 KLUE 검색기를 감싸세요. 한국어 질문이 들어가면 gpt-4o-mini 가 한국어로 유사 질문을 만들어 줍니다.

확장 질문 로깅을 켜서 어떤 한국어 변형 질문이 만들어지는지 직접 눈으로 확인하세요.

In [23]:
# Step F. Multi-Query Retrieval (KLUE)

import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever_klue = MultiQueryRetriever.from_llm(
    retriever=db_klue.as_retriever(search_kwargs={"k": 3}),
    llm=ChatOpenAI(model="gpt-4o", temperature=0),
)

docs_mq_klue = multi_query_retriever_klue.invoke(TEST_Q_KLUE)
# 문서 수가 3보다 많이 나오는 게 정상이다. 확장 질문 3개가 각각 3개씩 찾아온 뒤
# 중복만 제거해서 합치기 때문 (합집합). 이게 "검색 폭을 넓힌다" 의 실체다.
print(f"\n검색된 문서 수: {len(docs_mq_klue)}  (단일 질문이면 3개였을 것)")
print("---")
print(docs_mq_klue[0].page_content[:300])


INFO:langchain.retrievers.multi_query:Generated queries: ['1. 한국에서 해킹으로 인해 리플이 포함된 은행 계좌의 수는 몇 개입니까?', '2. 국내에서 리플이 해킹된 계좌의 총 수는 어떻게 되나요?', '3. 한국 내에서 해킹 피해를 입은 리플 보유 계좌의 수는 얼마인가요?']



검색된 문서 수: 6  (단일 질문이면 3개였을 것)
---
국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 200여명의 계좌에서 3월20일부터 30일까지 3억원어치의 리플이 도난당했다. 국내에서 발생한 가상화폐 해킹 사건으로는 이례적인 규모다. 200여명의 계좌에서 빠져나간 리플은 두 개의


### Step G. HyDE 직접 구현

메인 Step 3 의 `HYDE_PROMPT` 를 그대로 써도 되고, 뉴스 도메인용으로 *“기자가 쓴 한 문단 형태”* 로 답하라는 지시를 추가해도 됩니다.

`hyde_retrieve_klue(question, k=3)` 함수를 만들고 `db_klue` 위에서 동작하도록 하세요.

In [24]:
# Step G. HyDE 직접 구현 (KLUE)

# HYDE_PROMPT 도 메인 것을 그대로 재사용한다 (변수는 하나만 바꾼다는 같은 이유).
# ┌─ [관측용 · langfuse] ─ 여기부터 한 줄은 실습 로직이 아니다 ──────────────
# │  이 데코레이터는 함수가 하는 일을 바꾸지 않는다. 함수가 호출될 때
# │  '어떤 인자를 받아 무엇을 돌려줬는지' 를 langfuse 서버로 보내, 나중에 웹 화면에서
# │  흐름을 트리로 펼쳐 볼 수 있게 표시만 해두는 것이다. 지워도 결과는 똑같다.
# └──────────────────────────────────────────────────────────────────────
@observe(name="21_HyDE검색(KLUE)")
def hyde_retrieve_klue(question, k=3):
    """질문 -> LLM 이 지어낸 가상의 답변 -> 그 가상 답변을 임베딩해서 검색"""
    # 왜 이게 통하냐면: 질문과 문서는 문장 형태가 서로 달라서(질문 vs 서술문) 임베딩 거리가 멀다.
    # 가상 답변은 '서술문' 이라 진짜 기사 본문과 형태가 비슷해져서 더 잘 붙는다.
    hypothetical = hyde_generator.invoke({"question": question})
    return db_klue.similarity_search(hypothetical, k=k), hypothetical

docs_hyde_klue, hyp_klue = hyde_retrieve_klue(TEST_Q_KLUE)
print("가상 답변(HyDE):\n", hyp_klue[:300], "\n---")
print("검색된 문서 수:", len(docs_hyde_klue))
print("첫 문서:", docs_hyde_klue[0].page_content[:200])

# 가상 답변이 사실과 다를 수도 있다는 게 HyDE 의 핵심 포인트다.
# 틀려도 상관없다 — 그 문장을 '검색어' 로만 쓰지, 답변으로 쓰지 않는다.
print("\n(참고) 이 가상 답변이 사실인지 아닌지는 중요하지 않다. 검색 미끼로만 쓴다.")


가상 답변(HyDE):
 국내에서 해킹으로 인해 리플이 들어간 통장이 몇 개인지는 정확한 통계가 공개되지 않았기 때문에 확실한 숫자를 제시하기는 어렵습니다. 그러나 일반적으로 암호화폐 거래소나 개인 지갑이 해킹당하는 경우, 피해 규모와 관련된 정보는 금융 당국이나 관련 기관에서 조사하여 발표하는 경우가 많습니다. 따라서 이러한 사건이 발생했을 때, 피해자 수나 피해 규모에 대한 정보는 공식 발표를 통해 확인할 수 있습니다. 해킹 사건의 특성상 피해 규모는 사건의 성격, 해킹의 범위, 그리고 보안 조치의 수준에 따라 크게 달라질 수 있습니다. 따라서 관련 기 
---
검색된 문서 수: 3
첫 문서: 국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20

(참고) 이 가상 답변이 사실인지 아닌지는 중요하지 않다. 검색 미끼로만 쓴다.


### Step H. Multilingual Cross-encoder Reranker

메인 Step 4 에서 이미 `BAAI/bge-reranker-v2-m3` 같은 다국어 reranker 를 사용하고 있습니다. 추가 실습에서는:

- 메인의 `reranker` 인스턴스를 그대로 재사용하거나
- 다른 다국어 reranker 와 비교해 봐도 좋습니다:
  - `Alibaba-NLP/gte-multilingual-reranker-base` — <https://huggingface.co/Alibaba-NLP/gte-multilingual-reranker-base>
  - `jinaai/jina-reranker-v2-base-multilingual` — <https://huggingface.co/jinaai/jina-reranker-v2-base-multilingual>

`rerank_klue(query, docs, top_k=3)` 함수를 만드세요. (메인 Step 4 의 `rerank` 와 동일 구조)

In [25]:
# Step H. Multilingual Cross-encoder Reranker (KLUE)

# 메인 Step 4 에서 이미 올려둔 BAAI/bge-reranker-v2-m3 인스턴스를 그대로 재사용한다.
# 같은 모델을 두 번 로드하면 GPU 메모리만 두 배 먹고 얻는 게 없다.
reranker_klue = reranker

# ┌─ [관측용 · langfuse] ─ 여기부터 한 줄은 실습 로직이 아니다 ──────────────
# │  이 데코레이터는 함수가 하는 일을 바꾸지 않는다. 함수가 호출될 때
# │  '어떤 인자를 받아 무엇을 돌려줬는지' 를 langfuse 서버로 보내, 나중에 웹 화면에서
# │  흐름을 트리로 펼쳐 볼 수 있게 표시만 해두는 것이다. 지워도 결과는 똑같다.
# └──────────────────────────────────────────────────────────────────────
@observe(name="22_리랭킹(KLUE)")
def rerank_klue(query, docs, top_k=3):
    """넓게 검색해온 docs 를 cross-encoder 로 다시 점수화해 상위 top_k 만 남긴다"""
    # 임베딩 검색은 질문과 문서를 '따로' 벡터로 만들어 거리만 재는데(빠르지만 대충),
    # cross-encoder 는 (질문, 문서) 를 한 쌍으로 같이 넣어 관련도를 직접 매긴다(느리지만 정밀).
    # 그래서 넓게 거른 다음(post-retrieval) 쓰는 게 맞다. 처음부터 전체 문서에 쓰면 너무 느리다.
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker_klue.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:top_k]]

candidates_klue = db_klue.as_retriever(search_kwargs={"k": 10}).invoke(TEST_Q_KLUE)
_pairs = [(TEST_Q_KLUE, d.page_content) for d in candidates_klue]
_scores_before = reranker_klue.predict(_pairs)
top3_klue = rerank_klue(TEST_Q_KLUE, candidates_klue, top_k=3)

print(f"후보 {len(candidates_klue)}개 -> Reranker 로 상위 3개 선별")
# 순위가 실제로 바뀌었는지 눈으로 확인한다. 안 바뀌었으면 리랭커를 넣은 의미가 없다.
print("\n[검색 순서대로의 리랭커 점수] 값이 뒤죽박죽이면 = 원래 순위가 엉망이었다는 뜻")
for _i, _s in enumerate(_scores_before):
    print(f"  검색 {_i+1}등 -> 리랭커 점수 {_s:.4f}")
print("\n최상위 문서:", top3_klue[0].page_content[:200])


후보 10개 -> Reranker 로 상위 3개 선별

[검색 순서대로의 리랭커 점수] 값이 뒤죽박죽이면 = 원래 순위가 엉망이었다는 뜻
  검색 1등 -> 리랭커 점수 0.9748
  검색 2등 -> 리랭커 점수 0.0039
  검색 3등 -> 리랭커 점수 0.0000
  검색 4등 -> 리랭커 점수 0.0000
  검색 5등 -> 리랭커 점수 0.0000
  검색 6등 -> 리랭커 점수 0.0000
  검색 7등 -> 리랭커 점수 0.0000
  검색 8등 -> 리랭커 점수 0.0000
  검색 9등 -> 리랭커 점수 0.0000
  검색 10등 -> 리랭커 점수 0.0000

최상위 문서: 국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20


### Step I. Advanced RAG 체인 (넓게 → Rerank → LLM)

메인 Step 5 흐름과 동일.
1. `db_klue.as_retriever(search_kwargs={"k": 10}).invoke(question)` 로 후보 10개
2. `rerank_klue(question, candidates, top_k=3)` 로 좁힘
3. `RAG_PROMPT` + `llm` 으로 답변 생성

함수가 `(answer, top_docs)` 둘 다 반환하도록 만들어 두면 다음 평가 단계에서 그대로 씁니다.

In [26]:
# Step I. Advanced RAG 체인 (넓게 -> Rerank -> LLM)

# ┌─ [관측용 · langfuse] ─ 여기부터 한 줄은 실습 로직이 아니다 ──────────────
# │  이 데코레이터는 함수가 하는 일을 바꾸지 않는다. 함수가 호출될 때
# │  '어떤 인자를 받아 무엇을 돌려줬는지' 를 langfuse 서버로 보내, 나중에 웹 화면에서
# │  흐름을 트리로 펼쳐 볼 수 있게 표시만 해두는 것이다. 지워도 결과는 똑같다.
# └──────────────────────────────────────────────────────────────────────
@observe(name="23_Advanced_RAG_KLUE(전체)")
def advanced_rag_klue(question):
    # 1) 후보를 넓게 검색 (k=10). Naive 는 여기서 바로 3개만 뽑아 썼다.
    candidates = db_klue.as_retriever(search_kwargs={"k": 10}).invoke(question)

    # 2) Cross-encoder 로 진짜 관련도를 다시 매겨 상위 3개만 남긴다
    #    KorQuAD 쪽(Step 5)과 똑같이 retriever 타입으로 감싸서, 두 도메인의 트레이스를
    #    langfuse 화면에서 같은 모양으로 나란히 비교할 수 있게 한다.
    # ┌─ [관측용 · langfuse] ─ 아래 with 한 줄만 관측용이다 ──────────────
    # │  함수로 안 쪼갠 '코드 구간' 을 하나의 단계로 묶어 기록하는 방법.
    # │  as_type="retriever" 로 지정하면 langfuse 가 이걸 '검색 단계' 로 분류해서
    # │  어떤 문서가 근거로 들어갔는지 본문까지 보기 좋게 보여준다.
    # │  안쪽의 rerank(...) 호출이 실제 실습 코드이고, with 와 span.update 는 기록용이다.
    # └──────────────────────────────────────────────────────────────────
    with lf_client.start_as_current_observation(name="후보10개→상위3개(KLUE)", as_type="retriever") as span:
        top = rerank_klue(question, candidates, top_k=3)
        span.update(input={"질문": question, "후보수": len(candidates)},
                    output=[{"title": d.metadata.get("title"),
                             "page_content": d.page_content[:200]} for d in top])

    # 3) 그 3개를 컨텍스트로 넣어 답변 생성
    context = format_docs(top)
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": context, "question": question})
    return answer, top

ans_adv_klue, ctx_adv_klue = advanced_rag_klue(TEST_Q_KLUE)
print("Q:", TEST_Q_KLUE)
print("정답(ground truth):", ground_truths_klue[0])
print("\nAdvanced RAG 답변:\n", ans_adv_klue)
print("\n[근거로 쓴 문서 3개]")
for _i, _d in enumerate(ctx_adv_klue):
    print(f"  {_i+1}. ({_d.metadata['title']}) {_d.page_content[:100]}...")


Q: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?
정답(ground truth): 두 개

Advanced RAG 답변:
 두 개의 계좌입니다.

[근거로 쓴 문서 3개]
  1. (또 해킹당한 가상화폐) 국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에...
  2. (‘보안 구멍’ 드러난 인터넷익스플로러 ...국내 PC 76% ‘해킹 경보’) 마이크로소프트(MS)의 웹브라우저 인터넷익스플로러(IE)의 모든 버전(6~11)이 해킹에 무방비로 노출됐다. 국내 PC 이용자 가운데 IE를 웹브라우저로 쓰는 비중은 76%에 달한...
  3. (“부르는 게 값” … 추석 승차권·콘서트·고궁 표까지 정상가의 최대 10배...암표 거래 판쳐도 … 온라인 장터는 ‘무법지대’) 직장인 서아린 씨는 22일 서울 잠실종합운동장 보조경기장에서 열리는 록스타 본 조비의 내한공연 표를 사기 위해 각종 관람권을 거래하는 사이트인 ‘티켓베이’를 찾았다. 이미 동난 지...


### Step J. RAGAS 로 Naive vs Advanced 비교

메인 Step 6/7 흐름을 KLUE-MRC 변수(`_klue`) 로 옮겨 동일하게 수행하세요.

1. 20개 질문 각각을 Naive / Advanced 파이프라인에 돌려 답변과 컨텍스트 수집
2. `Dataset.from_dict({...})` 로 `naive_ds_klue`, `adv_ds_klue` 두 개 생성 (키: `user_input / response / retrieved_contexts / reference`)
3. `evaluate(..., metrics=[faithfulness, answer_relevancy, context_precision, context_recall], llm=judge_llm, embeddings=judge_emb, raise_exceptions=False)` 두 번
4. 평균표로 비교

메인의 KorQuAD 결과와 점수가 어떻게 다른지 옆에 같이 적어두면 학습 효과가 큽니다.

In [27]:
# Step J. RAGAS 로 Naive vs Advanced 비교 (KLUE-MRC)

# 1) 20개 질문을 두 파이프라인에 각각 돌려 답변과 컨텍스트를 모은다
naive_answers_klue, naive_contexts_klue = [], []
adv_answers_klue,   adv_contexts_klue   = [], []

for _i, q in enumerate(questions_klue):
    ctx = naive_retriever_klue.invoke(q)
    a = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(ctx), "question": q})
    naive_answers_klue.append(a)
    naive_contexts_klue.append([d.page_content for d in ctx])

    a2, ctx2 = advanced_rag_klue(q)
    adv_answers_klue.append(a2)
    adv_contexts_klue.append([d.page_content for d in ctx2])
    print(f"  {_i+1}/{len(questions_klue)} 완료")

# 2) RAGAS 가 먹는 형태로 Dataset 만들기
naive_ds_klue = Dataset.from_dict({
    "user_input":         questions_klue,
    "response":           naive_answers_klue,
    "retrieved_contexts": naive_contexts_klue,
    "reference":          ground_truths_klue,
})
adv_ds_klue = Dataset.from_dict({
    "user_input":         questions_klue,
    "response":           adv_answers_klue,
    "retrieved_contexts": adv_contexts_klue,
    "reference":          ground_truths_klue,
})

# 3) 채점. answer_relevancy 한국어 패치는 Step 7 에서 이미 적용해둔 게 그대로 살아 있다
#    (metrics 객체가 전역이라 프롬프트 수정이 유지된다).
print("\n=== KLUE Naive RAG 채점 ===")
naive_result_klue = evaluate(naive_ds_klue, metrics=metrics,
                             llm=judge_llm, embeddings=judge_emb,
                             raise_exceptions=False)
print("=== KLUE Advanced RAG 채점 ===")
adv_result_klue = evaluate(adv_ds_klue, metrics=metrics,
                           llm=judge_llm, embeddings=judge_emb,
                           raise_exceptions=False)

# 4) 비교표
naive_df_klue = naive_result_klue.to_pandas()
adv_df_klue   = adv_result_klue.to_pandas()

compare_klue = pd.concat([summary(naive_df_klue, "Naive RAG"),
                          summary(adv_df_klue,   "Advanced RAG")], axis=1)
print("\n===== KLUE-MRC (뉴스) =====")
print(compare_klue.round(3))
print("\nDelta (Advanced - Naive):")
print((compare_klue["Advanced RAG"] - compare_klue["Naive RAG"]).round(3))

# 5) 메인 실습(KorQuAD, 위키) 결과와 나란히 놓고 본다 — 이게 이번 추가 실습의 목적이다
print("\n\n===== 도메인 비교: KorQuAD(위키) vs KLUE-MRC(뉴스) =====")
_both = pd.concat([
    summary(naive_df,      "KorQuAD Naive"),
    summary(adv_df,        "KorQuAD Adv"),
    summary(naive_df_klue, "KLUE Naive"),
    summary(adv_df_klue,   "KLUE Adv"),
], axis=1)
print(_both.round(3))


  1/20 완료


  2/20 완료


  3/20 완료


  4/20 완료


  5/20 완료


...

  18/20 완료


  19/20 완료


  20/20 완료

=== KLUE Naive RAG 채점 ===


=== KLUE Advanced RAG 채점 ===



===== KLUE-MRC (뉴스) =====
                   Naive RAG  Advanced RAG
faithfulness           0.600         0.725
answer_relevancy       0.220         0.218
context_precision      0.717         0.900
context_recall         0.850         0.900

Delta (Advanced - Naive):
faithfulness         0.125
answer_relevancy    -0.001
context_precision    0.183
context_recall       0.050
dtype: float64


===== 도메인 비교: KorQuAD(위키) vs KLUE-MRC(뉴스) =====
                   KorQuAD Naive  KorQuAD Adv  KLUE Naive  KLUE Adv
faithfulness               0.675        0.850       0.600     0.725
answer_relevancy           0.208        0.184       0.220     0.218
context_precision          0.692        0.892       0.717     0.900
context_recall             0.750        0.900       0.850     0.900


### Step K. (선택) 좀 더 큰 샘플로 통계적 신뢰도 확보

질문 20개로는 표본 분산이 커서 Naive vs Advanced 차이가 우연일 수도 있습니다. 토큰 비용이 허용된다면 50~100문항으로 늘려 paired t-test 같은 간단한 통계 검정으로 차이가 유의한지 확인해 보세요.

참고: `scipy.stats.ttest_rel(naive_df["faithfulness"], adv_df["faithfulness"])`

In [28]:
# Step K. paired t-test 로 "차이가 우연인지" 검정하기

from scipy import stats

# 질문 수를 50~100 으로 늘리는 게 원래 안내지만, 오늘은 시간과 토큰 비용 때문에
# 질문 20개 그대로 두고 대신 "문항별 점수" 를 짝지어 검정한다.
# ★ 그래서 N=20 이고, 표본이 작다는 한계는 그대로 남아 있다. 이 점을 알고 결과를 읽어야 한다.
# paired(대응표본) 를 쓰는 이유: 같은 질문을 두 파이프라인이 각각 푼 것이라
# 질문 난이도라는 공통 요인을 상쇄할 수 있어서, 그냥 평균 비교보다 민감하다.

def paired_report(naive_df_, adv_df_, label):
    print(f"\n===== {label} (N={len(naive_df_)}) =====")
    for col in ["faithfulness", "answer_relevancy",
                "context_precision", "context_recall"]:
        a = naive_df_[col].astype(float).fillna(0.0)
        b = adv_df_[col].astype(float).fillna(0.0)
        diff = b.mean() - a.mean()
        # 두 열이 완전히 똑같으면(예: recall 이 둘 다 1.0 으로 포화) t 검정이 nan 을 뱉는다.
        if (b - a).abs().sum() == 0:
            print(f"  {col:20s} Δ={diff:+.3f}   두 파이프라인 점수가 완전히 동일 (검정 불가)")
            continue
        t, p = stats.ttest_rel(b, a)
        verdict = "유의미한 차이" if p < 0.05 else "우연일 수 있음 (차이 없다고 봐야 함)"
        print(f"  {col:20s} Δ={diff:+.3f}   t={t:+.3f}  p={p:.4f}  -> {verdict}")

paired_report(naive_df,      adv_df,      "KorQuAD (위키)")
paired_report(naive_df_klue, adv_df_klue, "KLUE-MRC (뉴스)")

print("\n[읽는 법] p < 0.05 여야 '진짜 좋아졌다' 고 말할 수 있다.")
print("         Δ 가 커 보여도 p 가 크면 20문항짜리 표본이 흔들린 것일 뿐이다.")



===== KorQuAD (위키) (N=20) =====
  faithfulness         Δ=+0.175   t=+2.101  p=0.0493  -> 유의미한 차이
  answer_relevancy     Δ=-0.024   t=-0.701  p=0.4919  -> 우연일 수 있음 (차이 없다고 봐야 함)
  context_precision    Δ=+0.200   t=+2.505  p=0.0215  -> 유의미한 차이
  context_recall       Δ=+0.150   t=+1.831  p=0.0828  -> 우연일 수 있음 (차이 없다고 봐야 함)

===== KLUE-MRC (뉴스) (N=20) =====
  faithfulness         Δ=+0.125   t=+1.751  p=0.0961  -> 우연일 수 있음 (차이 없다고 봐야 함)
  answer_relevancy     Δ=-0.001   t=-0.028  p=0.9776  -> 우연일 수 있음 (차이 없다고 봐야 함)
  context_precision    Δ=+0.183   t=+2.371  p=0.0285  -> 유의미한 차이
  context_recall       Δ=+0.050   t=+1.000  p=0.3299  -> 우연일 수 있음 (차이 없다고 봐야 함)

[읽는 법] p < 0.05 여야 '진짜 좋아졌다' 고 말할 수 있다.
         Δ 가 커 보여도 p 가 크면 20문항짜리 표본이 흔들린 것일 뿐이다.


### 마지막 Quiz — 직접 답을 적어보세요

1. **도메인 비교**: KorQuAD(위키) 와 KLUE-MRC(뉴스) 결과에서 4지표 중 가장 크게 달라진 건 무엇이었나요? 뉴스 기사의 어떤 특성(시점 표현, 인용, 숫자 등) 때문이라고 보이나요?
2. **Advanced 효과**: KLUE-MRC 에서도 Naive → Advanced 개선폭이 컸나요? KorQuAD 와 같았나요, 달랐나요?
3. **`is_impossible` 케이스**: Step B 에서 답할 수 없는 질문을 의도적으로 섞어 평가하면 어떤 지표가 가장 망가질까요? (실험해 보면 더 좋음)
4. (선택) 같은 파이프라인을 **MIRACL ko** 로 옮기면 어떤 차이가 있을지 예상해 보세요.

### 마지막 Quiz — 내가 적은 답

먼저 두 도메인 결과를 한 표에 놓고 본다 (위 Step J 출력).

| 지표 | KorQuAD Naive | KorQuAD Adv | KLUE Naive | KLUE Adv |
|---|---|---|---|---|
| faithfulness | 0.675 | **0.850** | 0.600 | **0.725** |
| answer_relevancy | 0.208 | 0.184 | 0.220 | 0.218 |
| context_precision | 0.692 | 0.892 | 0.717 | 0.900 |
| context_recall | 0.750 | 0.900 | 0.850 | 0.900 |

---

**1. 도메인 비교 — 4지표 중 가장 크게 달라진 건?**

**`faithfulness` 다.** 그런데 Naive 상태에서는 두 도메인이 0.650 vs 0.700 으로 별 차이가 없다.
크게 갈린 건 **Advanced 로 올렸을 때**다.

- KorQuAD(위키): 0.675 -> 0.850 (**+0.175**, p=0.0493 로 유의미)
- KLUE(뉴스): 0.600 -> 0.725 (**+0.125**, p=0.0961 로 단정 불가)

개선폭 자체는 KLUE 도 작지 않은데(+0.125), 문항별 편차가 커서 통계적으로는 "차이 없다" 쪽이다.
즉 위키에서는 지어내기가 줄었다고 말할 수 있고, 뉴스에서는 **아직 말할 수 없다**.

**왜 그럴까 — 내 추측은 문서 길이다.** Step B 에서 찍어본 KLUE 뉴스 기사는 평균 1063 토큰
(최소 474, 최대 2123)으로 상당히 길다. 리랭커가 상위 3개를 잘 골라줘도, 그 3개가 각각 길기 때문에
컨텍스트 안에 답과 무관한 문장이 여전히 잔뜩 남는다. 위키 쪽은 500 토큰 단위로 잘라 넣어서
고른 3개가 훨씬 촘촘하다. 즉 **"좋은 문서를 고르는 것" 과 "짧고 집중된 컨텍스트를 주는 것" 은 다른 문제**이고,
faithfulness 는 후자에 더 민감한 것으로 보인다.
뉴스 특성 중 인용문·숫자·시점 표현이 많다는 점도 같은 방향으로 작용했을 것 같은데,
이건 오늘 따로 확인하지 못해서 추측으로만 적어둔다.

---

**2. Advanced 효과 — KLUE 에서도 개선폭이 컸나?**

**아니다. 지표에 따라 갈렸고, 이게 오늘 가장 중요한 발견이다.**

Step K 의 paired t-test 결과를 그대로 옮기면:

| 지표 | KorQuAD | KLUE |
|---|---|---|
| context_precision | +0.200 (p=0.0215) **유의미** | +0.183 (p=0.0285) **유의미** |
| faithfulness | +0.175 (p=0.0493) **유의미** | +0.125 (p=0.0961) 차이 없음 |
| context_recall | +0.150 (p=0.0828) 차이 없음 | +0.050 (p=0.3299) 차이 없음 |
| answer_relevancy | -0.024 (p=0.4919) 차이 없음 | -0.001 (p=0.9776) 차이 없음 |

**두 도메인 모두에서 통계적으로 살아남은 지표는 `context_precision` 하나뿐이다.**

여기서 배운 것: **리랭커가 "검색을 개선한다" 는 효과는 도메인을 바꿔도 재현된다.**
반면 그것이 "답변 품질 개선" 으로 이어지느냐는 도메인에 따라 다르다.
평균값만 봤다면 KLUE 도 faithfulness 가 +0.125 올랐으니 "좋아졌다" 고 썼을 텐데,
p=0.0961 은 20문항이 흔들린 것일 수 있다는 뜻이다. **Δ 만 보고 결론 내면 안 된다**는 걸 숫자로 확인했다.

---

**3. `is_impossible` 케이스를 섞으면 어떤 지표가 가장 망가질까?**

**`context_recall` 이 먼저 망가진다.**
이 지표는 "정답(reference)의 내용이 검색된 컨텍스트 안에 들어 있느냐" 를 재는데,
`is_impossible=True` 인 샘플은 `answers["text"]` 가 비어 있다. 비교할 정답 자체가 없으니
채점이 성립하지 않는다. Step B 에서 이걸 걸러낸 이유가 그것이고, 오늘 데이터에서는
5841건 중 **1833건(약 31%)** 이 그런 케이스였다. 3분의 1을 안 거르고 넣었으면 표가 통째로 흔들렸을 것이다.

**그다음은 `faithfulness` 다.** 답할 수 없는 질문인데 LLM 이 "모르겠습니다" 대신 뭔가 지어내면
그 주장은 문서로 뒷받침되지 않으므로 점수가 떨어진다. 다만 이건 지표가 고장 난 게 아니라
**지표가 제대로 일한 결과**다 — 오히려 환각 탐지에는 이 케이스를 일부러 섞는 게 좋은 테스트가 된다.

`context_precision` 은 정답 문서라는 게 애초에 없으니 의미가 흐려지고,
`answer_relevancy` 는 답변에서 질문을 역추론하는 방식이라 "모르겠습니다" 같은 답에는 낮게 나올 것이다.

---

**4. (선택) MIRACL ko 로 옮기면?**

직접 돌려보지는 못해서 예상만 적는다.

- MIRACL 은 정답 텍스트가 아니라 **문서 단위 관련도 판정(qrels)** 형태라, 오늘처럼
  `reference` 에 정답 문자열을 넣는 방식을 그대로 쓸 수 없다. 평가 코드를 손봐야 할 것이다.
- 코퍼스가 오늘(위키 847개, 뉴스 299개)과는 비교가 안 되게 크다.
  그러면 오늘 0.9 에서 포화된 `context_recall` 이 훨씬 낮게 나올 것이고,
  **Naive 와 Advanced 의 차이도 그때 비로소 제대로 보일 것 같다.**
  오늘 recall 이 두 도메인 다 p>0.05 로 "차이 없음" 이 나온 건 DB 가 작아서일 가능성이 크다.


## 마치며

이번 실습에서는 한국어 QA 벤치마크 위에서 다음을 진행했습니다.

- **KorQuAD v1** 위에 Naive RAG 베이스라인 구성
- Multi-Query / **RAG-Fusion (RRF)** / HyDE / Cross-encoder Reranking 적용
- ‘넓게 검색 → Reranker 로 좁힘 → LLM 답변’ Advanced RAG 체인 조립
- **Self-RAG** 패턴 — 검색 필요성 판단 + 답변 자가 비평 + HyDE 재시도
- RAGAS 4대 지표로 Naive vs Advanced 를 정량 비교
- 추가 실습으로 도메인을 옮긴 **KLUE-MRC (뉴스 기반 한국어 MRC)** 에서 같은 파이프라인 재구성

**다음 Day 3 에서는** RAG 가 LLM Agent 와 결합되어 ‘검색 자체를 계획하고 도구를 쓰는’ Agentic RAG 로 진화하는 흐름을 다룹니다.

## 오늘의 회고

### 1. 오늘 실제로 만든 것

KorQuAD(위키) 위에 Naive RAG 를 세우고, 거기에 네 가지를 단계적으로 얹어 Advanced RAG 로 키웠다.
그리고 같은 파이프라인을 KLUE-MRC(뉴스) 로 통째로 옮겨 도메인이 바뀌면 무엇이 달라지는지 봤다.

| 단계 | 무엇을 | 어디에 붙나 |
|---|---|---|
| Multi-Query | 질문 하나를 여러 표현으로 늘려 검색 폭을 넓힘 | pre-retrieval |
| RAG-Fusion (RRF) | 쿼리별 검색 결과를 **순위** 기준으로 합침 | pre-retrieval |
| HyDE | 가상의 답변을 만들어 그걸로 검색 | pre-retrieval |
| Cross-Encoder Rerank | 넓게 뽑은 후보를 정밀하게 다시 줄 세움 | post-retrieval |
| Self-RAG | 검색이 필요한지 판단 + 답변을 스스로 비평 | 전체 흐름 제어 |

### 2. 개념에서 오늘 확실해진 것

**RRF 는 점수가 아니라 순위를 더한다.** 처음엔 왜 굳이 `1/(k+rank)` 같은 걸 쓰나 했는데,
검색기마다 유사도 점수의 스케일이 제각각이라 그대로 더하면 한 쪽이 판을 먹어버린다.
순위는 스케일이 없으니 공정하게 합칠 수 있다. 실제로 돌려보니 확장 질문 4개가 **전부** 같은 문서를
1등으로 뽑은 경우 점수 0.06531 로 최상위가 됐다. "여러 쿼리가 공통으로 찾아낸 문서" 가 위로 올라온다는
말이 이 숫자로 눈에 보였다.

**k=60 은 완충 장치다.** 1등이 1/61, 2등이 1/62 라서 둘 차이가 거의 없다. 한 쿼리가 1등으로 밀어붙인
문서보다, 여러 쿼리가 2~3등으로 공통 지목한 문서가 이길 수 있게 만드는 장치다.

**리랭커가 왜 post-retrieval 인가.** 임베딩은 문서 벡터를 미리 계산해두니 빠르지만 대충 본다.
크로스인코더는 (질문, 문서) 를 같이 넣어 즉석에서 재니 정밀하지만 느리다. 그래서 전체 문서에 쓸 수 없고,
넓게 거른 뒤에 쓴다. 오늘 셀별 실행 시간을 따로 재보니
**모델을 올리는 데(Step 4 첫 셀)만 302초**가 걸렸는데,
정작 후보 10개를 재정렬하고 답변까지 만드는 셀(Step 5)은 **2.7초**에 끝났다.
(이 두 숫자는 셀 출력에는 안 남는 값이라 실행하면서 따로 잰 것이고,
 2.7초에는 검색과 LLM 답변 생성도 포함이라 리랭커 단독 시간은 이보다 더 짧다.)
"리랭커는 느리다" 는 말은 **전체 문서를 훑을 때** 얘기지, 10개 정도면 비싸지 않다는 걸 숫자로 확인했다.

**HyDE 의 가상 답변은 사실이 아니어도 된다.** 질문과 문서는 문장 형태가 서로 달라(질문 vs 서술문)
임베딩 거리가 멀다. 가상 답변은 서술문이라 진짜 본문과 형태가 비슷해져 잘 붙는다.
그 문장을 **검색어로만** 쓰고 답변으로는 안 쓰기 때문에, 내용이 틀려도 상관없다.

### 3. 오늘 걸려 넘어진 것 다섯 가지 (이게 제일 남는다)

**(1) `answer_relevancy` 가 한국어에서 고장 나 있었다.**
이 지표는 "답변만 보고 질문을 거꾸로 만들어" 원래 질문과 유사도를 잰다. 그런데 ragas 0.2.10 의 역질문
프롬프트는 예시가 전부 영어라, 한국어 답변을 주면 **영어 질문**을 만들어낸다. 한국어 질문 vs 영어 역질문이라
유사도가 바닥으로 가고 심하면 음수(-0.0166)까지 나온다. `adapt_prompts` 에는 'korean' 이 아예 없었다.
처방은 역질문 프롬프트 지시문에 "응답과 같은 언어로 쓰라" 한 줄을 직접 못 박는 것이었다.
이 패치를 검증할 때(어제 따로 만든 확인용 스크립트), 문장으로 된 긴 답변은 **0.928** 까지 올랐지만
**짧은 답변은 0.198 로 여전히 낮았다.** 짧은 답은 주어가 빠져서 역추론이 흐려지니 낮은 게 맞는 것이다.
오늘 KorQuAD 본실험 값은 Naive 0.208 / Advanced 0.184 로, **0.928 쪽이 아니라 0.198 쪽에 붙어 있다.**
KorQuAD 정답이 평균 6.0자짜리 한 단어라 답변도 짧게 나오기 때문이다.
즉 오늘의 낮은 점수는 **버그가 남은 게 아니라, 정답이 한 단어라서 정당하게 낮은 것**이다.
(패치가 실제로 걸렸다는 건 Step 7 셀 출력의 `[패치] ...` 줄로 확인할 수 있다.)
이 패치를 안 했으면 비교표의 그 열은 통째로 '언어 불일치 노이즈' 였을 것이고,
나는 그걸 모른 채 "Advanced 가 나빠졌네" 라고 잘못 읽었을 것이다.

**(2) Self-RAG 의 자가 비평이 맞는 답을 틀렸다고 했다.**
최종 답변 "대중교통체계입니다" 는 문서에 분명히 있는 내용인데, 비평 LLM 이 두 번 다 `NOT_SUPPORTED`
판정을 내리고 HyDE 재검색까지 갔다. 답이 주어도 서술도 없는 한 단어라, 심판이 "이게 문서로 뒷받침되나"
를 판정할 근거 자체가 부족했던 것으로 보인다.
(1)번과 뿌리가 같다 — **한국어 추출형 QA의 짧은 정답이 LLM 심판을 흔든다.**
LLM-as-Judge 를 쓸 때는 심판이 보는 입력이 판정 가능한 형태인지부터 봐야 한다는 걸 배웠다.

**(3) 안내문대로 변수명을 썼으면 실험이 망가질 뻔했다.**
추가 실습 Step B 안내는 `context_docs` 를 만들라고 되어 있는데, 그건 메인 실습(KorQuAD)에서 이미 쓰는
이름이다. 그대로 썼으면 KorQuAD 쪽 변수를 덮어써서 **도메인 비교 자체가 불가능**해진다.
그래서 전부 `_klue` 접미사를 붙였고, Step C 에서 `db` 와 `db_klue` 의 문서 개수를 둘 다 출력해
덮어쓰기가 안 일어났는지 확인하도록 했다.

**(4) 그리고 그 확인 출력이 진짜 사고를 잡아냈다 — 두 Vector DB 가 같은 컬렉션이었다.**
Step C 를 안내대로 `db_klue = Chroma(embedding_function=embedding)` 로 만들고 돌렸더니,
확인 출력이 이렇게 나왔다.

```
db_klue 적재 완료 — 문서 1563개
메인 db(KorQuAD) 는 그대로 — 문서 1563개
```

**두 DB 의 문서 수가 똑같다.** KorQuAD 청크 1264개 + KLUE 뉴스 299개 = 1563.
`Chroma` 는 컬렉션 이름을 안 주면 기본값 `"langchain"` 을 쓰는데, 메모리 클라이언트가 그 이름을
공유하기 때문에 **변수만 두 개였고 실제 저장소는 하나**였던 것이다.
그래서 뉴스 질문을 던져도 위키 문서가 후보로 딸려 나오는, 도메인 비교가 통째로 오염된 상태였다.

무서운 건 **에러가 한 줄도 안 났다**는 점이다. 적재도 됐고, 검색도 됐고, RAGAS 표도 멀쩡한 숫자로 나왔다.
개수를 찍어보지 않았으면 그대로 제출하고 "뉴스 도메인은 이렇더라" 하고 틀린 결론을 적었을 것이다.
고친 방법은 `collection_name="klue_mrc"` 를 명시적으로 주는 것이고,
다시는 조용히 지나가지 않도록 `assert` 두 줄(컬렉션 이름이 다른지, 문서 수가 뉴스 개수와 맞는지)을 박아뒀다.
이 사고 때문에 실행을 처음부터 한 번 더 돌렸다. (오염된 1차 실행 결과는 기록용으로 따로 보관했다.)

**(5) 노트북이 GitHub 에서 빨간 경고를 띄우고 있었다 — "Could not render".**
Step 7 과 Step J 의 RAGAS 채점 출력 아래에 렌더링 실패 경고가 4곳 떠 있었다.
`evaluate()` 가 그리는 **진행률 표시줄이 ipywidgets** 라서 생긴 일이다. 그 출력은 두 벌로 저장되는데,
`application/vnd.jupyter.widget-view+json`(위젯)은 model_id 만 갖고 있고 실제 상태는
노트북 metadata 의 `widgets` 에 있어야 한다. 그런데 나는 이 노트북을 주피터 화면이 아니라
스크립트(nbclient)로 돌렸기 때문에 그 상태가 저장되지 않았다. 뷰어는 위젯을 먼저 그리려다
상태를 못 찾고 경고를 띄운 것이다.
처방은 **그 진행률 출력을 통째로 지우는 것**이다.
위젯이 남긴 글자 대체본은 `Evaluating: 0%| | 0/80` 인데(80 = 질문 20개 x 지표 4개 = 채점 80건),
진행 상황은 그 뒤 위젯 통신으로만 갱신되기 때문에 **이 대체본은 첫 프레임에서 멈춰 있다.**
실제로는 80건을 다 채점했는데 화면에는 0% 로 남는 셈이라, 남겨두면 오히려 사실과 어긋난다.
점수 출력은 별도라 그대로 남는다.
(100% 로 고쳐 적을 수도 있었지만 그건 **기록된 적 없는 출력을 지어내는 것**이라 하지 않았다.
 진짜 막대가 필요하면 tqdm 을 글자 모드로 강제해서 다시 돌려야 하고, 그건 시간이 없어 못 했다.)
★교훈: **내 화면에서 멀쩡해도 남의 화면(GitHub)에서는 깨질 수 있다.** 제출물은 올라간 뒤의
모습으로도 한 번 봐야 한다. 이건 코드가 틀린 게 아니라 '전달 매체' 에서 생기는 문제라 놓치기 쉽다.

### 4. 비교 실험에서 지킨 원칙

KLUE 로 옮길 때 **프롬프트(RAG_PROMPT, HYDE_PROMPT)와 리랭커 인스턴스는 메인 것을 그대로 재사용**했다.
뉴스용으로 프롬프트를 손보고 싶은 유혹이 있었지만, 그러면 점수가 달라졌을 때
"도메인이 달라서인지, 프롬프트를 바꿔서인지" 를 구분할 수 없다. 변수는 한 번에 하나만 바꾼다.

### 5. 도메인을 바꿔보고 알게 된 것

위키(KorQuAD) 에서 만든 파이프라인을 그대로 뉴스(KLUE-MRC) 로 옮겨 다시 쟀다. 결론부터 적으면
**두 도메인 모두에서 통계적으로 살아남은 개선은 `context_precision` 하나뿐이었다.**

| 지표 | KorQuAD | KLUE |
|---|---|---|
| context_precision | +0.200 (p=0.0215) **유의미** | +0.183 (p=0.0285) **유의미** |
| faithfulness | +0.175 (p=0.0493) **유의미** | +0.125 (p=0.0961) 차이 없음 |
| context_recall | +0.150 (p=0.0828) 차이 없음 | +0.050 (p=0.3299) 차이 없음 |
| answer_relevancy | -0.024 (p=0.4919) 차이 없음 | -0.001 (p=0.9776) 차이 없음 |

**"리랭커가 검색을 개선한다" 는 도메인을 넘어 재현됐지만, 그게 답변 품질로 이어지는지는 도메인을 탔다.**
평균만 봤으면 KLUE 도 faithfulness 가 +0.125 올랐으니 "여기서도 좋아졌다" 고 썼을 것이다.
p=0.0961 이면 20문항 표본이 흔들린 것일 수 있다는 뜻이라 단정하면 안 된다.
Step K 를 넣지 않았으면 나는 그 문장을 그냥 썼을 것이다.

차이가 난 이유로 짐작하는 건 **문서 길이**다. KLUE 뉴스 기사는 평균 1063 토큰인데
위키 쪽은 500 토큰으로 잘라 넣었다. 좋은 문서 3개를 골라도 각각이 길면 컨텍스트에 잡음이 남는다.
**"좋은 문서를 고르는 것" 과 "짧고 집중된 컨텍스트를 주는 것" 은 다른 문제**라는 것으로 이해했다.
(확인은 못 했다. 다음에 뉴스도 청크로 잘라 넣고 다시 재보면 알 수 있을 것 같다.)

### 6. 이 실험의 한계 (믿으면 안 되는 부분)

- **질문 20개, seed 하나.** ±0.05 정도 차이는 표본이 흔들린 것일 수 있다.
  그래서 Step K 에서 문항별 점수를 짝지어 paired t-test 로 확인했다.
- **★ 이건 추측이 아니라 실측이다.** 위 (4)번 사고 때문에 똑같은 코드를 처음부터 두 번 돌리게 됐는데,
  KorQuAD 결과가 두 번 다르게 나왔다.

  | 지표 | 1차 | 2차 | 3차(제출본) |
  |---|---|---|---|
  | faithfulness | 0.650 -> 0.800 | 0.650 -> 0.850 | 0.675 -> 0.850 |
  | answer_relevancy | 0.206 -> 0.181 | 0.271 -> 0.174 | 0.208 -> 0.184 |
  | context_precision | 0.692 -> 0.892 | 0.692 -> 0.883 | 0.692 -> 0.892 |
  | context_recall | 0.750 -> 0.900 | 0.750 -> 0.900 | 0.750 -> 0.900 |

  코드도 데이터도 seed 도 같은데 숫자가 움직였다. 채점하는 LLM(gpt-4o-mini)이 `temperature=0` 이어도
  완전히 결정적이지는 않기 때문이다. 특히 answer_relevancy 의 Naive 값이 0.206 / 0.271 / 0.208 로 벌어졌다.
  반면 context_recall 은 세 번 다 0.750 -> 0.900 으로 똑같았다.
  **즉 0.05 안팎의 차이를 놓고 "좋아졌다/나빠졌다" 를 말하면 안 된다**는 걸 우연히 증명하게 됐다.
  나중에 진짜 결론을 내려면 seed 를 바꿔 여러 번 돌린 평균으로 봐야 한다.
- Step K 도 질문 수를 늘린 게 아니라 **같은 20개**를 대응표본으로 검정한 것이라, N=20 의 한계는 그대로다.
- `context_recall` 은 DB 가 작으면(오늘 KorQuAD unique context 847개) 쉽게 포화된다.
  수만 문서 규모에서 다시 재봐야 진짜 차이가 보일 것이다.

### 7. 한 줄 요약

어제 배운 **"코드가 돌아간다 != 내가 재려던 걸 쟀다"** 가 오늘 두 번 더 반복됐다.
RAGAS 는 에러 없이 숫자를 뱉었지만 언어 불일치로 그 열이 의미가 없었고,
Vector DB 는 두 개인 줄 알았는데 실은 하나여서 도메인 비교가 오염돼 있었다.
**둘 다 에러가 아니라 "그럴듯한 숫자" 로 나타났다.**

그래서 오늘 얻은 실천 규칙은 이거다 —
**만든 것이 내가 만들려던 것인지, 숫자 하나로 찍어서 확인하고 넘어간다.**
문서 개수, 컬렉션 이름, 검색된 문서 수 같은 것들. 오늘은 그 한 줄이 제출을 살렸다.
